# Combined Climate Analysis

ETCCDI indices, warm spell duration, and surface moisture budget for the African continent under SSP5-8.5, SSP5-3.4-OS, and SAI scenarios using CESM2-WACCM6.


In [ ]:
import os
import glob
import re
import gc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
import pickle
from scipy import stats

# xclim imports for standardized ETCCDI climate indices
from xclim.indices import (
    growing_degree_days,
    warm_spell_duration_index,
    wetdays,
    max_n_day_precipitation_amount,
    maximum_consecutive_wet_days,
    maximum_consecutive_dry_days,
    daily_pr_intensity,
    prcptot,
)
from xclim.core.calendar import percentile_doy

warnings.filterwarnings('ignore')
print("Imports complete (including xclim)")


In [ ]:
# Configuration
prec_data_dir = "/home/temitope/paper/prec_data/"
temp_data_dir = "/home/temitope/paper/peekshaving_data/"
tmax_data_dir = "/home/temitope/paper/"
output_dir = "combined_analysis_plots/"
os.makedirs(output_dir, exist_ok=True)

africa_bounds = {'lat_min': -35, 'lat_max': 37, 'lon_min': -20, 'lon_max': 52}

analysis_periods = {
    'early_warming_15C': (2015, 2034),
    'early_warming_20C': (2024, 2044),
    'overshoot_period': (2060, 2079),
    'recovery_period': (2080, 2099)
}

# Base temperature for GDD (10C is standard for warm-season crops: maize, sorghum, millet)
# Reference: FAO Crop Information - Maize; Wikipedia GDD
BASE_TEMP_GDD = 10.0

# Optimal Growing Days (OGD) thresholds
# Lower threshold: 10C - minimum for germination/growth (FAO, 2024)
# Upper threshold: 35C - above this, heat stress reduces yields
# References:
#   - Lobell et al. (2011) Nature Climate Change: yield losses above 30C in Sub-Saharan Africa
#   - Sánchez et al. (2014): upper optimum 30-35C for maize
#   - PMC7913793: 'Maize leaf growth increases at 10-35C, declines above 35C'
#   - Crafts-Brandner & Law (2000): Rubisco activity reduced above 35C
TEMP_MIN_OGD = 10.0
TEMP_MAX_OGD = 35.0

metrics_config = {
    'mean_temperature': {'label': 'Mean Temperature', 'units': '°C', 'cmap_type': 'temp_increase'},
    'annual_gdd': {'label': 'Annual GDD', 'units': '°C-days', 'cmap_type': 'temp_increase'},
    'gdd_intensity': {'label': 'GDD Intensity', 'units': '°C-days/day', 'cmap_type': 'temp_increase'},
    'optimal_growing_days': {'label': 'Optimal Growing Days', 'units': 'days', 'cmap_type': 'precip_increase'},
    'wsdi': {'label': 'Warm Spell Duration Index', 'units': 'days', 'cmap_type': 'temp_increase'},
    'r1mm': {'label': 'Wet Days (R1mm)', 'units': 'days', 'cmap_type': 'precip_increase'},
    'rx5day': {'label': 'Max 5-day Precip (Rx5day)', 'units': 'mm', 'cmap_type': 'precip_increase'},
    'cwd': {'label': 'Consecutive Wet Days', 'units': 'days', 'cmap_type': 'precip_increase'},
    'cdd': {'label': 'Consecutive Dry Days', 'units': 'days', 'cmap_type': 'precip_decrease'},
    'sdii': {'label': 'Precip Intensity (SDII)', 'units': 'mm/day', 'cmap_type': 'precip_increase'},
    'prcptot': {'label': 'Total Precipitation', 'units': 'mm', 'cmap_type': 'precip_increase'}
}

# IPCC AR6 African Regions (approximate boundaries)
# Reference: https://www.ipcc.ch/report/ar6/wg1/chapter/atlas/
IPCC_REGIONS_AFRICA = {
    'SAH': {'name': 'Sahara', 'lat': (18, 30), 'lon': (-20, 33)},
    'WAF': {'name': 'West Africa', 'lat': (4, 18), 'lon': (-18, 15)},
    'CAF': {'name': 'Central Africa', 'lat': (-10, 8), 'lon': (8, 27)},
    'NEAF': {'name': 'N.E. Africa', 'lat': (4, 18), 'lon': (22, 52)},
    'SEAF': {'name': 'S.E. Africa', 'lat': (-12, 4), 'lon': (27, 52)},
    'WSAF': {'name': 'W.S. Africa', 'lat': (-35, -10), 'lon': (8, 24)},
    'ESAF': {'name': 'E.S. Africa', 'lat': (-35, -10), 'lon': (24, 42)},
    'MDG': {'name': 'Madagascar', 'lat': (-26, -12), 'lon': (42, 52)},
    'MED': {'name': 'Mediterranean', 'lat': (30, 37), 'lon': (-10, 40)}
}

# Monsoonal regions for moisture budget analysis
MONSOONAL_REGIONS = {
    'West Africa': {'lat': (4, 18), 'lon': (-18, 15), 'abbrev': 'WAF'},
    'Central Africa': {'lat': (-10, 8), 'lon': (8, 27), 'abbrev': 'CAF'},
    'East Africa': {'lat': (-12, 12), 'lon': (27, 52), 'abbrev': 'EAF'},
    'Southern Africa': {'lat': (-35, -10), 'lon': (8, 42), 'abbrev': 'SAF'}
}

# Moisture budget data directory
flux_data_dir = '/home/temitope/paper/sensible_latent/'


## Utility Functions


In [ ]:
def classify_scenario(filename):
    """Classify file into scenario category."""
    fname = os.path.basename(filename)
    
    if 'historical' in fname.lower():
        return 'historical'
    elif 'feedback.15C' in fname or 'feedback.1.5C' in fname:
        if 'SSP5-8.5' in fname:
            return 'ssp585_15_feedback'
        else:
            return 'ssp534OS_15_feedback'
    elif 'feedback.20C' in fname or 'feedback.2.0C' in fname:
        return 'ssp534OS_20_feedback'
    elif 'SSP5-3.4OS' in fname or 'SSP5-3.4-OS' in fname:
        return 'ssp534OS'
    elif 'SSP5-8.5' in fname:
        return 'ssp585'
    return 'unknown'

def extract_ensemble(filename):
    """Extract ensemble identifier."""
    fname = os.path.basename(filename)
    patterns = [r'\.(\d{3}-\d{3})\.', r'\.(\d{3})\.']
    for pattern in patterns:
        match = re.search(pattern, fname)
        if match:
            return match.group(1)
    return 'ens001'

def get_file_info(file_list):
    """Get file info without loading data."""
    file_info = []
    for f in file_list:
        scenario = classify_scenario(f)
        ensemble = extract_ensemble(f)
        if scenario != 'unknown' and scenario != 'historical':
            file_info.append({'path': f, 'scenario': scenario, 'ensemble': ensemble})
    return file_info

def remove_duplicate_times(ds):
    """Remove duplicate time values from dataset."""
    _, unique_idx = np.unique(ds.time.values, return_index=True)
    unique_idx = np.sort(unique_idx)
    if len(unique_idx) < len(ds.time):
        n_dups = len(ds.time) - len(unique_idx)
        print(f"    Removing {n_dups} duplicate times...")
        ds = ds.isel(time=unique_idx)
    return ds


## Data Preparation Functions


In [ ]:
def prepare_temperature_data(ds, var_name):
    """Prepare temperature DataArray with proper units for xclim."""
    da = ds[var_name].copy()
    
    # Check if in Kelvin and convert to Celsius
    if float(da.max()) > 200:  # Likely Kelvin
        da = da - 273.15
    
    # Set units attribute (required by xclim)
    da.attrs['units'] = '°C'
    
    # Rename to standard CF name for xclim
    if var_name == 'TREFHT':
        da = da.rename('tas')
    elif var_name == 'TREFHTMX':
        da = da.rename('tasmax')
    
    return da

def prepare_precipitation_data(ds, var_name='PRECT'):
    """Prepare precipitation DataArray with proper units for xclim."""
    da = ds[var_name].copy()
    
    # CESM PRECT is in m/s, convert to mm/day
    da = da * 86400 * 1000
    
    # Set units attribute (required by xclim)
    da.attrs['units'] = 'mm/day'
    
    # Rename to standard CF name
    da = da.rename('pr')
    
    return da


## xclim Index Calculations (ETCCDI Standards)


In [ ]:
def compute_gdd_indices(filepath):
    """Compute GDD-based indices using xclim.
    
    Mean Temperature: Annual mean temperature
    GDD (Growing Degree Days): Annual sum of (T - Tbase) for days where T > Tbase
    GDD Intensity: GDD / number of growing days
    Optimal Growing Days (OGD): Days within suitable temperature range (10-35C)
    
    OGD References:
    - Lobell et al. (2011) Nature Climate Change: yield losses above 30C in SSA
    - Sanchez et al. (2014): upper optimum 30-35C for maize
    - PMC7913793: Maize growth optimal 10-35C
    """
    print(f"  Computing GDD/OGD: {os.path.basename(filepath)}")
    
    ds = xr.open_dataset(filepath, decode_times=False)
    ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
    ds = xr.decode_cf(ds)
    
    if 'TREFHT' not in ds.data_vars:
        ds.close()
        return None
    
    # Handle duplicate times
    ds = remove_duplicate_times(ds)
    
    tas = prepare_temperature_data(ds, 'TREFHT')
    lat = ds.lat.values
    lon = ds.lon.values
    
    # Compute annual mean temperature
    mean_temp = tas.resample(time='YS').mean(dim='time')
    
    # Compute GDD using xclim
    gdd = growing_degree_days(tas=tas, thresh=f'{BASE_TEMP_GDD} °C', freq='YS')
    
    years = gdd.time.dt.year.values
    
    # Compute GDD intensity (GDD per growing day)
    days_above = (tas > BASE_TEMP_GDD).resample(time='YS').sum(dim='time')
    
    with np.errstate(divide='ignore', invalid='ignore'):
        gdd_intensity = xr.where(days_above > 0, gdd / days_above, np.nan)
    
    # Compute Optimal Growing Days (OGD)
    # Days where temperature is within suitable range for crop growth
    optimal_days = ((tas >= TEMP_MIN_OGD) & (tas <= TEMP_MAX_OGD)).resample(time='YS').sum(dim='time')
    
    results = {
        'mean_temperature': mean_temp.values,
        'annual_gdd': gdd.values,
        'gdd_intensity': gdd_intensity.values,
        'optimal_growing_days': optimal_days.values,
        'years_gdd': years,
        'lat': lat,
        'lon': lon
    }
    
    ds.close()
    del tas, gdd, gdd_intensity, days_above, optimal_days, mean_temp
    gc.collect()
    
    return results


In [ ]:
def compute_precip_indices(filepath):
    """Compute precipitation indices using xclim (ETCCDI standards).
    
    R1mm: Number of wet days (pr >= 1mm)
    Rx5day: Maximum consecutive 5-day precipitation
    CWD: Maximum consecutive wet days
    CDD: Maximum consecutive dry days
    SDII: Simple daily intensity index (mean precip on wet days)
    PRCPTOT: Total precipitation from wet days
    """
    print(f"  Computing precip indices: {os.path.basename(filepath)}")
    
    ds = xr.open_dataset(filepath, decode_times=False)
    ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
    ds = xr.decode_cf(ds)
    
    if 'PRECT' not in ds.data_vars:
        ds.close()
        return None
    
    # Handle duplicate times
    ds = remove_duplicate_times(ds)
    
    pr = prepare_precipitation_data(ds, 'PRECT')
    lat = ds.lat.values
    lon = ds.lon.values
    
    # R1mm: Number of wet days (precip >= 1mm)
    r1mm = wetdays(pr=pr, thresh='1 mm/day', freq='YS')
    
    # Rx5day: Maximum 5-day precipitation
    rx5day = max_n_day_precipitation_amount(pr=pr, window=5, freq='YS')
    
    # CWD: Maximum consecutive wet days
    cwd = maximum_consecutive_wet_days(pr=pr, thresh='1 mm/day', freq='YS')
    
    # CDD: Maximum consecutive dry days
    cdd = maximum_consecutive_dry_days(pr=pr, thresh='1 mm/day', freq='YS')
    
    # SDII: Simple daily intensity index (mean precip on wet days)
    sdii = daily_pr_intensity(pr=pr, thresh='1 mm/day', freq='YS')
    
    # PRCPTOT: Total precipitation from wet days
    prcptot_vals = prcptot(pr=pr, thresh='1 mm/day', freq='YS')
    
    years = r1mm.time.dt.year.values
    
    results = {
        'r1mm': r1mm.values,
        'rx5day': rx5day.values,
        'cwd': cwd.values,
        'cdd': cdd.values,
        'sdii': sdii.values,
        'prcptot': prcptot_vals.values,
        'years_prec': years,
        'lat': lat,
        'lon': lon
    }
    
    ds.close()
    del pr, r1mm, rx5day, cwd, cdd, sdii, prcptot_vals
    gc.collect()
    
    return results

print("Precipitation indices computation function defined (xclim)")


In [ ]:
def compute_baseline_percentile(tmax_files, baseline_period=(2015, 2034)):
    """Compute 90th percentile of tasmax for each day of year using xclim.
    
    Uses xclim's percentile_doy for ETCCDI-compliant calculation with 15-day window.
    """
    print(f"Computing WSDI baseline thresholds from {baseline_period}...")
    
    # Find ssp585 files for baseline (before SAI intervention)
    baseline_files = [f for f in tmax_files 
                      if f['scenario'] == 'ssp585'][:2]  # Use first 2 ensembles
    
    if not baseline_files:
        print("  No baseline files found!")
        return None
    
    all_tasmax = []
    
    for finfo in baseline_files:
        print(f"  Loading baseline: {os.path.basename(finfo['path'])}")
        ds = xr.open_dataset(finfo['path'], decode_times=False)
        ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
        ds = xr.decode_cf(ds)
        
        if 'TREFHTMX' not in ds.data_vars:
            ds.close()
            continue
        
        # Handle duplicate times
        ds = remove_duplicate_times(ds)
        
        tasmax = prepare_temperature_data(ds, 'TREFHTMX')
        
        # Select baseline period
        tasmax_baseline = tasmax.sel(time=slice(f'{baseline_period[0]}-01-01', 
                                                 f'{baseline_period[1]}-12-31'))
        all_tasmax.append(tasmax_baseline)
        ds.close()
    
    if not all_tasmax:
        return None
    
    # Concatenate and take mean across realizations
    combined = xr.concat(all_tasmax, dim='realization')
    tasmax_combined = combined.mean('realization')
    
    # Ensure units are preserved (required by xclim)
    tasmax_combined.attrs['units'] = '°C'
    
    # Use xclim's percentile_doy with 15-day window
    print("  Computing 90th percentile with xclim percentile_doy...")
    tasmax_per = percentile_doy(tasmax_combined, per=90, window=15)
    
    # Ensure output has units attribute (required by warm_spell_duration_index)
    tasmax_per.attrs['units'] = '°C'
    
    print(f"  Baseline thresholds computed: shape {tasmax_per.shape}")
    
    del all_tasmax, combined, tasmax_combined
    gc.collect()
    
    return tasmax_per

print("Baseline percentile function defined (xclim)")


In [ ]:
def compute_wsdi_indices(filepath, tasmax_per):
    """Compute WSDI using xclim's warm_spell_duration_index.
    
    WSDI: Count of days in warm spells (6+ consecutive days > 90th percentile)
    Following ETCCDI standard.
    """
    print(f"  Computing WSDI: {os.path.basename(filepath)}")
    
    if tasmax_per is None:
        print("    No baseline thresholds available, skipping WSDI")
        return None
    
    ds = xr.open_dataset(filepath, decode_times=False)
    ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
    ds = xr.decode_cf(ds)
    
    if 'TREFHTMX' not in ds.data_vars:
        ds.close()
        return None
    
    # Handle duplicate times
    ds = remove_duplicate_times(ds)
    
    tasmax = prepare_temperature_data(ds, 'TREFHTMX')
    lat = ds.lat.values
    lon = ds.lon.values
    
    # Compute WSDI using xclim
    wsdi = warm_spell_duration_index(tasmax=tasmax, tasmax_per=tasmax_per, 
                                      window=6, freq='YS')
    
    years = wsdi.time.dt.year.values
    
    results = {
        'wsdi': wsdi.values,
        'years_wsdi': years,
        'lat': lat,
        'lon': lon
    }
    
    ds.close()
    del tasmax, wsdi
    gc.collect()
    
    return results

print("WSDI computation function defined (xclim)")


## Process All Data (One Ensemble at a Time)


In [ ]:
def process_all_data(temp_files, tmax_files, prec_files, tasmax_per):
    """Process all files, computing indices for each ensemble member separately.
    
    Metrics are computed for each ensemble member individually.
    Ensemble averaging happens AFTER metric computation in compute_period_means().
    """
    all_results = {}
    
    # Group files by scenario and ensemble
    def group_files(file_list):
        grouped = {}
        for f in file_list:
            key = (f['scenario'], f['ensemble'])
            grouped[key] = f['path']
        return grouped
    
    temp_grouped = group_files(temp_files)
    tmax_grouped = group_files(tmax_files)
    prec_grouped = group_files(prec_files)
    
    # Get all unique scenario/ensemble combinations
    all_keys = set(temp_grouped.keys()) | set(tmax_grouped.keys()) | set(prec_grouped.keys())
    
    for scenario, ensemble in sorted(all_keys):
        print(f"\nProcessing {scenario} / {ensemble}")
        
        if scenario not in all_results:
            all_results[scenario] = {}
        
        ens_results = {}
        
        # Process TREFHT (GDD, OGD, Mean Temperature)
        if (scenario, ensemble) in temp_grouped:
            results = compute_gdd_indices(temp_grouped[(scenario, ensemble)])
            if results:
                ens_results['mean_temperature'] = results['mean_temperature']
                ens_results['annual_gdd'] = results['annual_gdd']
                ens_results['gdd_intensity'] = results['gdd_intensity']
                ens_results['optimal_growing_days'] = results['optimal_growing_days']
                ens_results['years_gdd'] = results['years_gdd']
                ens_results['lat'] = results['lat']
                ens_results['lon'] = results['lon']
            del results
            gc.collect()
        
        # Process PRECT (precipitation indices)
        if (scenario, ensemble) in prec_grouped:
            results = compute_precip_indices(prec_grouped[(scenario, ensemble)])
            if results:
                for k in ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']:
                    ens_results[k] = results[k]
                ens_results['years_prec'] = results['years_prec']
                if 'lat' not in ens_results:
                    ens_results['lat'] = results['lat']
                    ens_results['lon'] = results['lon']
            del results
            gc.collect()
        
        # Process TREFHTMX (WSDI)
        if (scenario, ensemble) in tmax_grouped:
            results = compute_wsdi_indices(tmax_grouped[(scenario, ensemble)], tasmax_per)
            if results and 'wsdi' in results:
                ens_results['wsdi'] = results['wsdi']
                ens_results['years_wsdi'] = results['years_wsdi']
                if 'lat' not in ens_results:
                    ens_results['lat'] = results['lat']
                    ens_results['lon'] = results['lon']
            del results
            gc.collect()
        
        # Store combined years (use longest)
        all_years = []
        for ykey in ['years_gdd', 'years_prec', 'years_wsdi']:
            if ykey in ens_results and ens_results[ykey] is not None:
                all_years.append(ens_results[ykey])
        if all_years:
            ens_results['years'] = max(all_years, key=len)
        
        if ens_results:
            all_results[scenario][ensemble] = ens_results
    
    return all_results


## Merge Scenarios and Compute Period Means


In [ ]:
def get_years_for_metric(data_dict, metric):
    """Get the appropriate years array for a given metric."""
    if metric in ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days']:
        return data_dict.get('years_gdd', data_dict.get('years'))
    elif metric in ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']:
        return data_dict.get('years_prec', data_dict.get('years'))
    elif metric == 'wsdi':
        return data_dict.get('years_wsdi', data_dict.get('years'))
    return data_dict.get('years')

def merge_scenario_results(all_results, early_scenario, late_scenario, merged_name, split_year=2040):
    """Merge early and late scenario results."""
    if early_scenario not in all_results or late_scenario not in all_results:
        print(f"  Skipping {merged_name}: missing {early_scenario} or {late_scenario}")
        return
    
    merged = {}
    common_ens = set(all_results[early_scenario].keys()) & set(all_results[late_scenario].keys())
    
    for ens in common_ens:
        early = all_results[early_scenario][ens]
        late = all_results[late_scenario][ens]
        
        merged_ens = {'lat': early.get('lat'), 'lon': early.get('lon')}
        
        for metric in metrics_config.keys():
            if metric not in early or metric not in late:
                continue
            if early[metric] is None or late[metric] is None:
                continue
            
            early_years = get_years_for_metric(early, metric)
            late_years = get_years_for_metric(late, metric)
            
            if early_years is None or late_years is None:
                continue
            
            early_data = np.squeeze(early[metric])
            late_data = np.squeeze(late[metric])
            
            # Verify shapes match
            if early_data.shape[0] != len(early_years) or late_data.shape[0] != len(late_years):
                continue
            
            early_mask = early_years < split_year
            late_mask = late_years >= split_year
            
            merged_ens[metric] = np.concatenate([early_data[early_mask], late_data[late_mask]], axis=0)
            
            # Store merged years
            if metric in ['annual_gdd', 'gdd_intensity', 'optimal_growing_days']:
                merged_ens['years_gdd'] = np.concatenate([early_years[early_mask], late_years[late_mask]])
            elif metric in ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']:
                merged_ens['years_prec'] = np.concatenate([early_years[early_mask], late_years[late_mask]])
            elif metric == 'wsdi':
                merged_ens['years_wsdi'] = np.concatenate([early_years[early_mask], late_years[late_mask]])
        
        # Create combined years
        all_years = []
        for ykey in ['years_gdd', 'years_prec', 'years_wsdi']:
            if ykey in merged_ens:
                all_years.append(merged_ens[ykey])
        if all_years:
            merged_ens['years'] = max(all_years, key=len)
        
        if any(k in merged_ens for k in metrics_config.keys()):
            merged[ens] = merged_ens
    
    if merged:
        all_results[merged_name] = merged
        print(f"  Created {merged_name} with {len(merged)} ensemble(s)")


In [ ]:
def compute_period_means(all_results, period_years):
    """Compute ensemble means for a period.
    
    This is where ensemble averaging happens - AFTER metrics have been
    computed for each individual ensemble member.
    
    For each metric:
    1. Extract temporal mean over the period for each ensemble member
    2. Then compute ensemble mean across all members
    """
    period_means = {}
    
    for scenario in all_results:
        scenario_data = {}
        
        for metric in metrics_config.keys():
            metric_vals = []  # Will hold period mean for each ensemble
            
            for ens, ens_results in all_results[scenario].items():
                if metric not in ens_results:
                    continue
                if ens_results[metric] is None:
                    continue
                
                years = get_years_for_metric(ens_results, metric)
                if years is None:
                    continue
                
                data = np.squeeze(ens_results[metric])
                
                if data.shape[0] != len(years):
                    continue
                
                # Step 1: Temporal mean for this ensemble member
                year_mask = (years >= period_years[0]) & (years <= period_years[1])
                if np.sum(year_mask) > 0:
                    ens_period_mean = np.nanmean(data[year_mask], axis=0)
                    metric_vals.append(ens_period_mean)
            
            # Step 2: Ensemble mean across all ensemble members
            if metric_vals:
                scenario_data[metric] = np.nanmean(metric_vals, axis=0)
        
        if scenario_data:
            period_means[scenario] = scenario_data
    
    return period_means


## Statistical Testing Functions


In [ ]:
def compute_ensemble_period_data(all_results, period_years):
    """Compute period means for each ensemble member (for statistical testing).
    
    Returns data structure with individual ensemble values for t-tests.
    """
    ensemble_data = {}
    
    for scenario in all_results:
        scenario_data = {}
        
        for metric in metrics_config.keys():
            metric_vals = []  # List of (lat, lon) arrays, one per ensemble
            
            for ens, ens_results in all_results[scenario].items():
                if metric not in ens_results:
                    continue
                if ens_results[metric] is None:
                    continue
                
                years = get_years_for_metric(ens_results, metric)
                if years is None:
                    continue
                
                data = np.squeeze(ens_results[metric])
                
                if data.shape[0] != len(years):
                    continue
                
                # Temporal mean for this ensemble member
                year_mask = (years >= period_years[0]) & (years <= period_years[1])
                if np.sum(year_mask) > 0:
                    ens_period_mean = np.nanmean(data[year_mask], axis=0)
                    metric_vals.append(ens_period_mean)
            
            if metric_vals:
                # Stack into (n_ensemble, lat, lon) array
                scenario_data[metric] = np.stack(metric_vals, axis=0)
        
        if scenario_data:
            ensemble_data[scenario] = scenario_data
    
    return ensemble_data

def compute_significance_map(data1, data2, alpha=0.05):
    """Compute pixel-wise significance using Welch's t-test.
    
    Parameters:
    -----------
    data1 : ndarray (n_ensemble, lat, lon)
        First dataset (e.g., baseline period)
    data2 : ndarray (n_ensemble, lat, lon)
        Second dataset (e.g., overshoot period)
    alpha : float
        Significance level (default 0.05)
    
    Returns:
    --------
    sig_mask : ndarray (lat, lon)
        Boolean mask where True = significant difference
    p_values : ndarray (lat, lon)
        P-values for each pixel
    """
    if data1 is None or data2 is None:
        return None, None
    
    # Get spatial dimensions
    if data1.ndim == 2:
        # Only one ensemble member - can't do t-test
        return None, None
    
    nlat, nlon = data1.shape[1], data1.shape[2]
    p_values = np.full((nlat, nlon), np.nan)
    
    for i in range(nlat):
        for j in range(nlon):
            vals1 = data1[:, i, j]
            vals2 = data2[:, i, j]
            
            # Remove NaN values
            valid1 = vals1[~np.isnan(vals1)]
            valid2 = vals2[~np.isnan(vals2)]
            
            # Need at least 2 values in each group for t-test
            if len(valid1) >= 2 and len(valid2) >= 2:
                try:
                    _, p = stats.ttest_ind(valid1, valid2, equal_var=False)  # Welch's t-test
                    p_values[i, j] = p
                except:
                    pass
    
    sig_mask = p_values < alpha
    return sig_mask, p_values

def compute_all_significance(ensemble_baseline, ensemble_future, scenarios_to_test):
    """Compute significance maps for all metrics and scenario comparisons.
    
    Parameters:
    -----------
    ensemble_baseline : dict
        Per-ensemble data for baseline period
    ensemble_future : dict
        Per-ensemble data for future period
    scenarios_to_test : list of tuples
        List of (baseline_scenario, future_scenario) pairs to test
    
    Returns:
    --------
    significance : dict
        Nested dict: significance[future_scenario][metric] = sig_mask
    """
    significance = {}
    
    for baseline_scen, future_scen in scenarios_to_test:
        if baseline_scen not in ensemble_baseline or future_scen not in ensemble_future:
            continue
        
        significance[future_scen] = {}
        
        for metric in metrics_config.keys():
            if metric not in ensemble_baseline[baseline_scen]:
                continue
            if metric not in ensemble_future[future_scen]:
                continue
            
            base_data = ensemble_baseline[baseline_scen][metric]
            future_data = ensemble_future[future_scen][metric]
            
            sig_mask, _ = compute_significance_map(base_data, future_data)
            if sig_mask is not None:
                significance[future_scen][metric] = sig_mask
    
    return significance

def compute_summary_statistics(period_means_baseline, period_means_future, 
                               ensemble_baseline, ensemble_future,
                               land_mask, scenarios):
    """Compute summary statistics table for all metrics.
    
    Returns DataFrame with:
    - Regional mean for baseline and future
    - Absolute and percent change
    - Significance (% of land area with p < 0.05)
    """
    results = []
    
    for baseline_scen, future_scen in scenarios:
        if baseline_scen not in period_means_baseline:
            continue
        if future_scen not in period_means_future:
            continue
        
        for metric in metrics_config.keys():
            cfg = metrics_config[metric]
            
            if metric not in period_means_baseline[baseline_scen]:
                continue
            if metric not in period_means_future[future_scen]:
                continue
            
            base_data = np.squeeze(period_means_baseline[baseline_scen][metric])
            future_data = np.squeeze(period_means_future[future_scen][metric])
            
            # Regional means (land only)
            base_mean = np.nanmean(base_data[land_mask])
            future_mean = np.nanmean(future_data[land_mask])
            
            # Changes
            abs_change = future_mean - base_mean
            pct_change = (abs_change / base_mean * 100) if base_mean != 0 else np.nan
            
            # Significance
            sig_pct = np.nan
            if (baseline_scen in ensemble_baseline and future_scen in ensemble_future and
                metric in ensemble_baseline[baseline_scen] and metric in ensemble_future[future_scen]):
                sig_mask, _ = compute_significance_map(
                    ensemble_baseline[baseline_scen][metric],
                    ensemble_future[future_scen][metric]
                )
                if sig_mask is not None:
                    sig_pct = np.sum(sig_mask & land_mask) / np.sum(land_mask) * 100
            
            results.append({
                'Scenario': future_scen,
                'Metric': cfg['label'],
                'Units': cfg['units'],
                'Baseline Mean': f'{base_mean:.2f}',
                'Future Mean': f'{future_mean:.2f}',
                'Abs Change': f'{abs_change:.2f}',
                'Pct Change': f'{pct_change:.1f}%',
                'Sig Area (%)': f'{sig_pct:.1f}%' if not np.isnan(sig_pct) else 'N/A'
            })
    
    return pd.DataFrame(results)

def plot_statistics_visualization(period_means_baseline, period_means_future,
                                   ensemble_baseline, ensemble_future,
                                   land_mask, scenarios, threshold_label, output_dir):
    """Create publication-quality statistical visualizations.
    
    Creates 3 figures:
    1. Bar chart comparing baseline - future for each metric
    2. Percent change with significance indicators
    3. Heatmap of significance (% area with p<0.05)
    """
    
    # Collect data for visualization
    viz_data = []
    
    for baseline_scen, future_scen in scenarios:
        if baseline_scen not in period_means_baseline:
            continue
        if future_scen not in period_means_future:
            continue
        
        # Friendly scenario names
        if future_scen == 'ssp585':
            scen_label = 'SSP5-8.5'
        elif future_scen == 'full_ssp534OS':
            scen_label = 'SSP5-3.4-OS'
        else:
            scen_label = future_scen
        
        for metric in metrics_config.keys():
            cfg = metrics_config[metric]
            
            if metric not in period_means_baseline[baseline_scen]:
                continue
            if metric not in period_means_future[future_scen]:
                continue
            
            base_data = np.squeeze(period_means_baseline[baseline_scen][metric])
            future_data = np.squeeze(period_means_future[future_scen][metric])
            
            base_mean = np.nanmean(base_data[land_mask])
            future_mean = np.nanmean(future_data[land_mask])
            base_std = np.nanstd(base_data[land_mask])
            future_std = np.nanstd(future_data[land_mask])
            
            abs_change = future_mean - base_mean
            pct_change = (abs_change / base_mean * 100) if base_mean != 0 else 0
            
            # Significance
            sig_pct = 0
            if (baseline_scen in ensemble_baseline and future_scen in ensemble_future and
                metric in ensemble_baseline.get(baseline_scen, {}) and 
                metric in ensemble_future.get(future_scen, {})):
                sig_mask, _ = compute_significance_map(
                    ensemble_baseline[baseline_scen][metric],
                    ensemble_future[future_scen][metric]
                )
                if sig_mask is not None:
                    sig_pct = np.sum(sig_mask & land_mask) / np.sum(land_mask) * 100
            
            viz_data.append({
                'scenario': scen_label,
                'metric': metric,
                'metric_label': cfg['label'],
                'units': cfg['units'],
                'baseline_mean': base_mean,
                'baseline_std': base_std,
                'future_mean': future_mean,
                'future_std': future_std,
                'abs_change': abs_change,
                'pct_change': pct_change,
                'sig_pct': sig_pct
            })
    
    df = pd.DataFrame(viz_data)
    
    if df.empty:
        print(f"  No data for {threshold_label} visualization")
        return
    
    scenarios_list = df['scenario'].unique()
    metrics_list = df['metric'].unique()
    n_metrics = len(metrics_list)
    
    # Color scheme
    colors = {'SSP5-8.5': '#d62728', 'SSP5-3.4-OS': '#2ca02c'}
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx, metric in enumerate(metrics_list):
        if idx >= len(axes):
            break
        ax = axes[idx]
        metric_df = df[df['metric'] == metric]
        
        x = np.arange(len(scenarios_list))
        width = 0.35
        
        # Get baseline (same for both scenarios)
        baseline_val = metric_df['baseline_mean'].iloc[0]
        baseline_std = metric_df['baseline_std'].iloc[0]
        
        # Baseline bars
        bars1 = ax.bar(x - width/2, [baseline_val]*len(scenarios_list), width, 
                       label='Baseline', color='#1f77b4', alpha=0.7,
                       yerr=[baseline_std]*len(scenarios_list), capsize=3)
        
        # Future bars for each scenario
        future_vals = [metric_df[metric_df['scenario']==s]['future_mean'].values[0] 
                       for s in scenarios_list]
        future_stds = [metric_df[metric_df['scenario']==s]['future_std'].values[0] 
                       for s in scenarios_list]
        future_colors = [colors.get(s, 'gray') for s in scenarios_list]
        
        bars2 = ax.bar(x + width/2, future_vals, width, 
                       color=future_colors, alpha=0.7,
                       yerr=future_stds, capsize=3)
        
        # Add significance stars
        for i, s in enumerate(scenarios_list):
            sig = metric_df[metric_df['scenario']==s]['sig_pct'].values[0]
            if sig > 50:  # More than 50% significant area
                ymax = max(baseline_val + baseline_std, future_vals[i] + future_stds[i])
                ax.text(x[i] + width/2, ymax * 1.02, '***', ha='center', fontsize=12, fontweight='bold')
            elif sig > 25:
                ymax = max(baseline_val + baseline_std, future_vals[i] + future_stds[i])
                ax.text(x[i] + width/2, ymax * 1.02, '**', ha='center', fontsize=12, fontweight='bold')
            elif sig > 10:
                ymax = max(baseline_val + baseline_std, future_vals[i] + future_stds[i])
                ax.text(x[i] + width/2, ymax * 1.02, '*', ha='center', fontsize=12, fontweight='bold')
        
        ax.set_ylabel(metric_df['units'].iloc[0])
        ax.set_title(metric_df['metric_label'].iloc[0], fontweight='bold', fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels(scenarios_list, fontsize=9)
        ax.tick_params(axis='y', labelsize=12)
        
        if idx == 0:
            ax.legend(['Baseline', 'Future'], loc='upper right', fontsize=8)
    
    # Hide empty subplots
    for idx in range(n_metrics, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'Baseline - Overshoot Period - {threshold_label} Threshold\n(Error bars: spatial std; *: >10%, **: >25%, ***: >50% significant area)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'{output_dir}/stats_barplot_{threshold_label}.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig, ax = plt.subplots(figsize=(18, 10))
    
    x = np.arange(n_metrics)
    width = 0.35
    
    for i, scen in enumerate(scenarios_list):
        scen_df = df[df['scenario'] == scen].set_index('metric')
        pct_changes = [scen_df.loc[m, 'pct_change'] if m in scen_df.index else 0 for m in metrics_list]
        sig_pcts = [scen_df.loc[m, 'sig_pct'] if m in scen_df.index else 0 for m in metrics_list]
        
        # Color bars by significance
        bar_colors = []
        for pct, sig in zip(pct_changes, sig_pcts):
            if sig > 25:
                bar_colors.append(colors.get(scen, 'gray'))
            else:
                bar_colors.append(colors.get(scen, 'gray') + '60')  # Transparent if not significant
        
        offset = (i - len(scenarios_list)/2 + 0.5) * width
        bars = ax.bar(x + offset, pct_changes, width, label=scen, 
                      color=colors.get(scen, 'gray'), alpha=0.8,
                      edgecolor='black', linewidth=0.5)
        
        # Add hatching for non-significant bars
        for bar, sig in zip(bars, sig_pcts):
            if sig <= 25:
                bar.set_hatch('//')
                bar.set_alpha(0.4)
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Climate Index', fontsize=15, fontweight='bold')
    ax.set_ylabel('Change (%)', fontsize=15, fontweight='bold')
    ax.set_title(f'Percent Change from Baseline to Overshoot - {threshold_label} Threshold\n(Hatched bars: <25% area significant)', 
                 fontsize=14, fontweight='bold')
    
    # Get metric labels
    metric_labels = [metrics_config[m]['label'] for m in metrics_list]
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=45, ha='right', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/stats_pct_change_{threshold_label}.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Create pivot table for heatmap
    heatmap_data = df.pivot(index='metric_label', columns='scenario', values='sig_pct')
    
    # Reorder rows to match metrics_config order
    ordered_labels = [metrics_config[m]['label'] for m in metrics_list if metrics_config[m]['label'] in heatmap_data.index]
    heatmap_data = heatmap_data.reindex(ordered_labels)
    
    im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Significant Area (%)', fontsize=11)
    
    # Add text annotations
    for i in range(len(heatmap_data.index)):
        for j in range(len(heatmap_data.columns)):
            val = heatmap_data.iloc[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > 50 else 'black'
                ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                        fontsize=10, fontweight='bold', color=text_color)
    
    ax.set_xticks(np.arange(len(heatmap_data.columns)))
    ax.set_yticks(np.arange(len(heatmap_data.index)))
    ax.set_xticklabels(heatmap_data.columns, fontsize=11)
    ax.set_yticklabels(heatmap_data.index, fontsize=10)
    ax.set_xlabel('Scenario', fontsize=15, fontweight='bold')
    ax.set_ylabel('Climate Index', fontsize=15, fontweight='bold')
    ax.set_title(f'Statistical Significance - {threshold_label} Threshold\n(% of land area with p < 0.05)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/stats_significance_heatmap_{threshold_label}.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  Saved statistics visualizations for {threshold_label}")

def plot_sai_statistics(period_means, ensemble_data, land_mask, output_dir):
    """Create publication-quality SAI impact statistical visualizations.
    
    Compares SAI scenarios - non-SAI scenarios during overshoot period.
    Creates figures showing SAI effectiveness across all metrics.
    """
    
    # Define SAI comparison pairs: (reference_scenario, sai_scenario, label)
    sai_comparisons = [
        ('ssp585', 'ssp585_1p5_feedback', 'SAI 1.5°C - SSP5-8.5'),
        ('full_ssp534OS', 'ssp534OS_1p5_feedback', 'SAI 1.5°C - SSP5-3.4-OS'),
        ('ssp585', 'ssp585_2p0_feedback', 'SAI 2.0°C - SSP5-8.5'),
        ('full_ssp534OS', 'ssp534OS_2p0_feedback', 'SAI 2.0°C - SSP5-3.4-OS'),
    ]
    
    # Collect data
    viz_data = []
    
    for ref_scen, sai_scen, comparison_label in sai_comparisons:
        if ref_scen not in period_means or sai_scen not in period_means:
            continue
        
        for metric in metrics_config.keys():
            cfg = metrics_config[metric]
            
            if metric not in period_means[ref_scen] or metric not in period_means[sai_scen]:
                continue
            
            ref_data = np.squeeze(period_means[ref_scen][metric])
            sai_data = np.squeeze(period_means[sai_scen][metric])
            
            ref_mean = np.nanmean(ref_data[land_mask])
            sai_mean = np.nanmean(sai_data[land_mask])
            ref_std = np.nanstd(ref_data[land_mask])
            sai_std = np.nanstd(sai_data[land_mask])
            
            abs_change = sai_mean - ref_mean
            pct_change = (abs_change / ref_mean * 100) if ref_mean != 0 else 0
            
            # Significance testing
            sig_pct = 0
            if (ref_scen in ensemble_data and sai_scen in ensemble_data and
                metric in ensemble_data.get(ref_scen, {}) and 
                metric in ensemble_data.get(sai_scen, {})):
                sig_mask, _ = compute_significance_map(
                    ensemble_data[ref_scen][metric],
                    ensemble_data[sai_scen][metric]
                )
                if sig_mask is not None:
                    sig_pct = np.sum(sig_mask & land_mask) / np.sum(land_mask) * 100
            
            viz_data.append({
                'comparison': comparison_label,
                'ref_scenario': ref_scen,
                'sai_scenario': sai_scen,
                'metric': metric,
                'metric_label': cfg['label'],
                'units': cfg['units'],
                'cmap_type': cfg['cmap_type'],
                'ref_mean': ref_mean,
                'ref_std': ref_std,
                'sai_mean': sai_mean,
                'sai_std': sai_std,
                'abs_change': abs_change,
                'pct_change': pct_change,
                'sig_pct': sig_pct
            })
    
    df = pd.DataFrame(viz_data)
    
    if df.empty:
        print("  No SAI data available for statistics")
        return
    
    comparisons_list = df['comparison'].unique()
    metrics_list = df['metric'].unique()
    n_metrics = len(metrics_list)
    n_comparisons = len(comparisons_list)
    
    # Color scheme for SAI comparisons
    colors = {
        'SAI 1.5°C - SSP5-8.5': '#1f77b4',
        'SAI 1.5°C - SSP5-3.4-OS': '#2ca02c',
        'SAI 2.0°C - SSP5-8.5': '#ff7f0e',
        'SAI 2.0°C - SSP5-3.4-OS': '#9467bd'
    }
    
    fig, ax = plt.subplots(figsize=(20, 10))
    
    x = np.arange(n_metrics)
    width = 0.2
    
    for i, comp in enumerate(comparisons_list):
        comp_df = df[df['comparison'] == comp].set_index('metric')
        pct_changes = [comp_df.loc[m, 'pct_change'] if m in comp_df.index else 0 for m in metrics_list]
        sig_pcts = [comp_df.loc[m, 'sig_pct'] if m in comp_df.index else 0 for m in metrics_list]
        
        offset = (i - n_comparisons/2 + 0.5) * width
        bars = ax.bar(x + offset, pct_changes, width, label=comp, 
                      color=colors.get(comp, 'gray'), alpha=0.85,
                      edgecolor='black', linewidth=0.5)
        
        # Add hatching for non-significant
        for bar, sig in zip(bars, sig_pcts):
            if sig <= 25:
                bar.set_hatch('//')
                bar.set_alpha(0.4)
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Climate Index', fontsize=15, fontweight='bold')
    ax.set_ylabel('SAI Effect - Change from Non-SAI (%)', fontsize=15, fontweight='bold')
    ax.set_title('SAI Impact on Climate Indices - Overshoot Period (2060-2079)\n(Negative = SAI reduces the index; Hatched = <25% area significant)', 
                 fontsize=14, fontweight='bold')
    
    metric_labels = [metrics_config[m]['label'] for m in metrics_list]
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=45, ha='right', fontsize=10)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/stats_sai_effect_pct.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Left: Percent change heatmap
    ax = axes[0]
    heatmap_pct = df.pivot(index='metric_label', columns='comparison', values='pct_change')
    ordered_labels = [metrics_config[m]['label'] for m in metrics_list if metrics_config[m]['label'] in heatmap_pct.index]
    heatmap_pct = heatmap_pct.reindex(ordered_labels)
    
    # Use diverging colormap centered at 0
    vmax = max(abs(heatmap_pct.values.min()), abs(heatmap_pct.values.max()))
    im = ax.imshow(heatmap_pct.values, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Change from Non-SAI (%)', fontsize=11)
    
    for i in range(len(heatmap_pct.index)):
        for j in range(len(heatmap_pct.columns)):
            val = heatmap_pct.iloc[i, j]
            if not np.isnan(val):
                text_color = 'white' if abs(val) > vmax*0.5 else 'black'
                ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                        fontsize=9, fontweight='bold', color=text_color)
    
    ax.set_xticks(np.arange(len(heatmap_pct.columns)))
    ax.set_yticks(np.arange(len(heatmap_pct.index)))
    ax.set_xticklabels(heatmap_pct.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(heatmap_pct.index, fontsize=10)
    ax.set_title('SAI Effect (% Change)', fontsize=12, fontweight='bold')
    
    # Right: Significance heatmap
    ax = axes[1]
    heatmap_sig = df.pivot(index='metric_label', columns='comparison', values='sig_pct')
    heatmap_sig = heatmap_sig.reindex(ordered_labels)
    
    im = ax.imshow(heatmap_sig.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Significant Area (%)', fontsize=11)
    
    for i in range(len(heatmap_sig.index)):
        for j in range(len(heatmap_sig.columns)):
            val = heatmap_sig.iloc[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > 50 else 'black'
                ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                        fontsize=9, fontweight='bold', color=text_color)
    
    ax.set_xticks(np.arange(len(heatmap_sig.columns)))
    ax.set_yticks(np.arange(len(heatmap_sig.index)))
    ax.set_xticklabels(heatmap_sig.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(heatmap_sig.index, fontsize=10)
    ax.set_title('Statistical Significance (p<0.05)', fontsize=12, fontweight='bold')
    
    plt.suptitle('SAI Impact Analysis - Overshoot Period (2060-2079)', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{output_dir}/stats_sai_heatmaps.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Categorize metrics
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    for ax_idx, (category, metric_subset, title) in enumerate([
        (axes[0], temp_metrics, 'Temperature-Related Indices'),
        (axes[1], precip_metrics, 'Precipitation-Related Indices')
    ]):
        ax = category
        subset_metrics = [m for m in metric_subset if m in metrics_list]
        
        x = np.arange(len(subset_metrics))
        width = 0.2
        
        for i, comp in enumerate(comparisons_list):
            comp_df = df[df['comparison'] == comp].set_index('metric')
            pct_changes = [comp_df.loc[m, 'pct_change'] if m in comp_df.index else 0 for m in subset_metrics]
            
            offset = (i - n_comparisons/2 + 0.5) * width
            ax.bar(x + offset, pct_changes, width, label=comp if ax_idx == 0 else '', 
                   color=colors.get(comp, 'gray'), alpha=0.85,
                   edgecolor='black', linewidth=0.5)
        
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
        ax.set_ylabel('SAI Effect (%)', fontsize=16, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        
        metric_labels = [metrics_config[m]['label'] for m in subset_metrics]
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels, rotation=45, ha='right', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
    
    axes[0].legend(loc='best', fontsize=8)
    plt.suptitle('SAI Impact by Index Category - Overshoot Period (2060-2079)', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'{output_dir}/stats_sai_by_category.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # Save SAI statistics table
    sai_summary = df[['comparison', 'metric_label', 'units', 'ref_mean', 'sai_mean', 
                      'abs_change', 'pct_change', 'sig_pct']].copy()
    sai_summary.columns = ['Comparison', 'Metric', 'Units', 'Non-SAI Mean', 'SAI Mean',
                           'Abs Change', 'Pct Change', 'Sig Area (%)']
    sai_summary['Non-SAI Mean'] = sai_summary['Non-SAI Mean'].apply(lambda x: f'{x:.2f}')
    sai_summary['SAI Mean'] = sai_summary['SAI Mean'].apply(lambda x: f'{x:.2f}')
    sai_summary['Abs Change'] = sai_summary['Abs Change'].apply(lambda x: f'{x:.2f}')
    sai_summary['Pct Change'] = sai_summary['Pct Change'].apply(lambda x: f'{x:.1f}%')
    sai_summary['Sig Area (%)'] = sai_summary['Sig Area (%)'].apply(lambda x: f'{x:.1f}%')
    sai_summary.to_csv(f'{output_dir}/summary_statistics_sai.csv', index=False)
    
    print("  Saved SAI statistics visualizations")

def plot_combined_statistics(period_means_15C_baseline, period_means_15C_future,
                             period_means_20C_baseline, period_means_20C_future,
                             ensemble_15C_baseline, ensemble_15C_future,
                             ensemble_20C_baseline, ensemble_20C_future,
                             land_mask, scenarios, output_dir):
    """Create combined 1.5°C and 2.0°C statistics visualizations.
    
    Generates figures that show both thresholds side-by-side for easy comparison.
    Splits metrics into Temperature (top row) and Precipitation (bottom row).
    """
    
    # Define metric categories
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    # Prepare data for both thresholds
    def prepare_viz_data(period_means_baseline, period_means_future, 
                         ensemble_baseline, ensemble_future):
        viz_data = []
        for baseline_scen, future_scen in scenarios:
            if baseline_scen not in period_means_baseline:
                continue
            if future_scen not in period_means_future:
                continue
            
            scen_label = 'SSP5-8.5' if future_scen == 'ssp585' else ('SSP5-3.4-OS' if future_scen == 'full_ssp534OS' else future_scen)
            
            for metric in metrics_config.keys():
                cfg = metrics_config[metric]
                if metric not in period_means_baseline[baseline_scen] or metric not in period_means_future[future_scen]:
                    continue
                
                base_data = np.squeeze(period_means_baseline[baseline_scen][metric])
                future_data = np.squeeze(period_means_future[future_scen][metric])
                
                base_mean = np.nanmean(base_data[land_mask])
                future_mean = np.nanmean(future_data[land_mask])
                base_std = np.nanstd(base_data[land_mask])
                future_std = np.nanstd(future_data[land_mask])
                
                abs_change = future_mean - base_mean
                pct_change = (abs_change / base_mean * 100) if base_mean != 0 else 0
                
                sig_pct = 0
                if (baseline_scen in ensemble_baseline and future_scen in ensemble_future and
                    metric in ensemble_baseline.get(baseline_scen, {}) and 
                    metric in ensemble_future.get(future_scen, {})):
                    sig_mask, _ = compute_significance_map(
                        ensemble_baseline[baseline_scen][metric],
                        ensemble_future[future_scen][metric]
                    )
                    if sig_mask is not None:
                        sig_pct = np.sum(sig_mask & land_mask) / np.sum(land_mask) * 100
                
                viz_data.append({
                    'scenario': scen_label,
                    'metric': metric,
                    'metric_label': cfg['label'],
                    'units': cfg['units'],
                    'baseline_mean': base_mean,
                    'baseline_std': base_std,
                    'future_mean': future_mean,
                    'future_std': future_std,
                    'abs_change': abs_change,
                    'pct_change': pct_change,
                    'sig_pct': sig_pct
                })
        return pd.DataFrame(viz_data)
    
    df_15C = prepare_viz_data(period_means_15C_baseline, period_means_15C_future,
                              ensemble_15C_baseline, ensemble_15C_future)
    df_20C = prepare_viz_data(period_means_20C_baseline, period_means_20C_future,
                              ensemble_20C_baseline, ensemble_20C_future)
    
    if df_15C.empty or df_20C.empty:
        print("  No data for combined visualization")
        return
    
    scenarios_list = df_15C['scenario'].unique()
    colors = {'SSP5-8.5': '#d62728', 'SSP5-3.4-OS': '#2ca02c'}
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    for col_idx, (df, threshold_label) in enumerate([(df_15C, '1.5°C'), (df_20C, '2.0°C')]):
        for row_idx, (metric_list, category_name) in enumerate([
            (temp_metrics, 'Temperature Indices'),
            (precip_metrics, 'Precipitation Indices')
        ]):
            ax = axes[row_idx, col_idx]
            
            available_metrics = [m for m in metric_list if m in df['metric'].values]
            n_metrics = len(available_metrics)
            
            if n_metrics == 0:
                ax.set_visible(False)
                continue
            
            x = np.arange(n_metrics)
            width = 0.35
            
            for scen_idx, scen in enumerate(scenarios_list):
                scen_df = df[df['scenario'] == scen].set_index('metric')
                
                baseline_vals = [scen_df.loc[m, 'baseline_mean'] if m in scen_df.index else 0 for m in available_metrics]
                future_vals = [scen_df.loc[m, 'future_mean'] if m in scen_df.index else 0 for m in available_metrics]
                future_stds = [scen_df.loc[m, 'future_std'] if m in scen_df.index else 0 for m in available_metrics]
                sig_pcts = [scen_df.loc[m, 'sig_pct'] if m in scen_df.index else 0 for m in available_metrics]
                
                offset = (scen_idx - len(scenarios_list)/2 + 0.5) * width
                
                # Baseline (lighter)
                ax.bar(x + offset - width/4, baseline_vals, width/2, 
                       color='#1f77b4', alpha=0.5, label='Baseline' if scen_idx == 0 else '')
                
                # Future
                bars = ax.bar(x + offset + width/4, future_vals, width/2,
                              color=colors.get(scen, 'gray'), alpha=0.8,
                              yerr=future_stds, capsize=2,
                              label=f'{scen}' if row_idx == 0 else '')
                
                # Add significance stars
                for i, (fv, fs, sig) in enumerate(zip(future_vals, future_stds, sig_pcts)):
                    ymax = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else fv + fs
                    if sig > 50:
                        ax.text(x[i] + offset + width/4, fv + fs + 0.02*ymax, '***', 
                                ha='center', fontsize=8, fontweight='bold')
                    elif sig > 25:
                        ax.text(x[i] + offset + width/4, fv + fs + 0.02*ymax, '**', 
                                ha='center', fontsize=8, fontweight='bold')
                    elif sig > 10:
                        ax.text(x[i] + offset + width/4, fv + fs + 0.02*ymax, '*', 
                                ha='center', fontsize=8, fontweight='bold')
            
            metric_labels = [metrics_config[m]['label'] for m in available_metrics]
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, rotation=30, ha='right', fontsize=9)
            ax.set_title(f'{threshold_label} - {category_name}', fontsize=11, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            
            if col_idx == 0:
                ax.set_ylabel('Value', fontsize=16, fontweight='bold')
            if row_idx == 0 and col_idx == 0:
                ax.legend(loc='upper right', fontsize=8)
    
    plt.suptitle('Baseline - Overshoot Period - Combined Thresholds\n(*: >10%, **: >25%, ***: >50% significant area)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'{output_dir}/stats_barplot_combined.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for col_idx, (df, threshold_label) in enumerate([(df_15C, '1.5°C'), (df_20C, '2.0°C')]):
        for row_idx, (metric_list, category_name) in enumerate([
            (temp_metrics, 'Temperature Indices'),
            (precip_metrics, 'Precipitation Indices')
        ]):
            ax = axes[row_idx, col_idx]
            
            available_metrics = [m for m in metric_list if m in df['metric'].values]
            n_metrics = len(available_metrics)
            
            if n_metrics == 0:
                ax.set_visible(False)
                continue
            
            x = np.arange(n_metrics)
            width = 0.35
            
            for scen_idx, scen in enumerate(scenarios_list):
                scen_df = df[df['scenario'] == scen].set_index('metric')
                pct_changes = [scen_df.loc[m, 'pct_change'] if m in scen_df.index else 0 for m in available_metrics]
                sig_pcts = [scen_df.loc[m, 'sig_pct'] if m in scen_df.index else 0 for m in available_metrics]
                
                offset = (scen_idx - len(scenarios_list)/2 + 0.5) * width
                bars = ax.bar(x + offset, pct_changes, width, 
                              color=colors.get(scen, 'gray'), alpha=0.8,
                              edgecolor='black', linewidth=0.5,
                              label=scen if row_idx == 0 and col_idx == 0 else '')
                
                # Hatch non-significant
                for bar, sig in zip(bars, sig_pcts):
                    if sig <= 25:
                        bar.set_hatch('//')
                        bar.set_alpha(0.4)
            
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
            metric_labels = [metrics_config[m]['label'] for m in available_metrics]
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, rotation=30, ha='right', fontsize=9)
            ax.set_title(f'{threshold_label} - {category_name}', fontsize=11, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            
            if col_idx == 0:
                ax.set_ylabel('Change (%)', fontsize=16, fontweight='bold')
    
    axes[0, 0].legend(loc='upper right', fontsize=9)
    plt.suptitle('Percent Change from Baseline - Combined Thresholds\n(Hatched: <25% area significant)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'{output_dir}/stats_pct_change_combined.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    fig = plt.figure(figsize=(18, 8))
    # Create gridspec with dedicated colorbar column
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.08)
    
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    cax = fig.add_subplot(gs[0, 2])
    
    for col_idx, (df, threshold_label) in enumerate([(df_15C, '1.5°C'), (df_20C, '2.0°C')]):
        ax = axes[col_idx]
        
        all_metrics = temp_metrics + precip_metrics
        available_metrics = [m for m in all_metrics if m in df['metric'].values]
        
        heatmap_data = df.pivot(index='metric_label', columns='scenario', values='sig_pct')
        ordered_labels = [metrics_config[m]['label'] for m in available_metrics if metrics_config[m]['label'] in heatmap_data.index]
        heatmap_data = heatmap_data.reindex(ordered_labels)
        
        im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)
        
        # Add text annotations
        for i in range(len(heatmap_data.index)):
            for j in range(len(heatmap_data.columns)):
                val = heatmap_data.iloc[i, j]
                if not np.isnan(val):
                    text_color = 'white' if val > 50 else 'black'
                    ax.text(j, i, f'{val:.1f}%', ha='center', va='center', 
                            fontsize=9, fontweight='bold', color=text_color)
        
        ax.set_xticks(np.arange(len(heatmap_data.columns)))
        ax.set_yticks(np.arange(len(heatmap_data.index)))
        ax.set_xticklabels(heatmap_data.columns, fontsize=10)
        ax.set_xlabel('Scenario', fontsize=16, fontweight='bold')
        ax.set_title(f'{threshold_label} Threshold', fontsize=12, fontweight='bold')
        
        # Only show y-axis labels on first heatmap
        if col_idx == 0:
            ax.set_yticklabels(heatmap_data.index, fontsize=9)
            ax.set_ylabel('Climate Index', fontsize=16, fontweight='bold')
        else:
            ax.set_yticklabels([])  # No labels on second heatmap
            ax.tick_params(axis='y', length=0)  # Hide tick marks too
        
        # Add horizontal line to separate temp and precip
        n_temp = len([m for m in temp_metrics if metrics_config[m]['label'] in heatmap_data.index])
        if n_temp > 0 and n_temp < len(heatmap_data.index):
            ax.axhline(y=n_temp - 0.5, color='white', linewidth=2)
    
    # Add colorbar in dedicated axis
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label('Significant Area (%)', fontsize=11)
    
    plt.suptitle('Statistical Significance (% of land area with p < 0.05) - Combined Thresholds', 
                 fontsize=14, fontweight='bold')
    plt.savefig(f'{output_dir}/stats_significance_heatmap_combined.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print("  Saved combined statistics visualizations")


## Plotting Functions


In [ ]:
import shapely.geometry as sgeom
from shapely.prepared import prep
import cartopy.io.shapereader as shpreader

def create_land_mask(lat, lon):
    """Create land mask using Natural Earth shapefile."""
    print("Creating land mask...")
    try:
        land_shp = shpreader.natural_earth(resolution='50m', category='physical', name='land')
        land_geom = [prep(r.geometry) for r in shpreader.Reader(land_shp).records()]
        lon_grid, lat_grid = np.meshgrid(lon, lat)
        mask = np.zeros_like(lon_grid, dtype=bool)
        for i in range(len(lat)):
            for j in range(len(lon)):
                pt = sgeom.Point(lon_grid[i,j], lat_grid[i,j])
                if any(g.contains(pt) for g in land_geom):
                    mask[i,j] = True
        print(f"  Land mask created: {np.sum(mask)} land points")
        return mask
    except Exception as e:
        print(f"  Warning: Could not create land mask: {e}")
        return np.ones((len(lat), len(lon)), dtype=bool)

def setup_africa_map(ax):
    """Setup Africa map with coastlines and borders."""
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, color='gray')
    ax.add_feature(cfeature.OCEAN, color='white')
    ax.set_extent([africa_bounds['lon_min'], africa_bounds['lon_max'],
                   africa_bounds['lat_min'], africa_bounds['lat_max']], ccrs.PlateCarree())

def create_rdbu_cmap():
    """Red-Blue diverging colormap (red=positive, blue=negative)."""
    colors = ['#053061','#2166ac','#4393c3','#92c5de','#d1e5f0','#f7f7f7',
              '#fddbc7','#f4a582','#d6604d','#b2182b','#67001f']
    return mcolors.LinearSegmentedColormap.from_list('rdbu', colors, N=256)

def create_burd_cmap():
    """Blue-Red diverging colormap (blue=positive, red=negative)."""
    colors = ['#67001f','#b2182b','#d6604d','#f4a582','#fddbc7','#f7f7f7',
              '#d1e5f0','#92c5de','#4393c3','#2166ac','#053061']
    return mcolors.LinearSegmentedColormap.from_list('burd', colors, N=256)

def get_diff_cmap(cmap_type):
    """Get appropriate diverging colormap for differences."""
    if cmap_type == 'precip_increase':  # More precip = blue (positive)
        return create_burd_cmap()
    return create_rdbu_cmap()  # More heat = red (positive)

def get_baseline_cmap(cmap_type):
    """Get appropriate colormap for baseline (absolute) values."""
    if cmap_type in ['precip_increase']:  # Wetness metrics - blue = high
        return 'YlGnBu'
    elif cmap_type == 'precip_decrease':  # CDD - more dry days = red
        return 'YlOrRd'
    else:  # Temperature metrics - red = high heat
        return 'YlOrRd'

def add_stippling(ax, lon, lat, sig_mask, land_mask, density=3):
    """Add stippling to show statistically significant areas.
    
    Parameters:
    -----------
    ax : matplotlib axis
    lon, lat : coordinate arrays
    sig_mask : boolean array where True = significant
    land_mask : boolean array for land points
    density : int, plot every nth point (lower = denser stippling)
    """
    if sig_mask is None:
        return
    
    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    
    # Get significant points (both significant AND on land)
    combined_mask = sig_mask & land_mask
    
    # Subsample for visibility
    sig_lons = lon_grid[combined_mask][::density]
    sig_lats = lat_grid[combined_mask][::density]
    
    # Plot stippling
    ax.scatter(sig_lons, sig_lats, s=8, c='black', marker='.', 
               transform=ccrs.PlateCarree(), alpha=0.6)

def add_ipcc_boundaries(ax, labels=True, linewidth=0.8, linestyle='--', color='black', alpha=0.6):
    """Add IPCC AR6 region boundaries to a cartopy axis.
    
    Parameters:
    -----------
    ax : matplotlib axis with cartopy projection
    labels : bool, whether to add region name labels
    linewidth : float, width of boundary lines
    linestyle : str, style of boundary lines
    color : str, color of boundary lines
    alpha : float, transparency of lines
    """
    for region_id, region_info in IPCC_REGIONS_AFRICA.items():
        lat_min, lat_max = region_info['lat']
        lon_min, lon_max = region_info['lon']
        
        # Draw rectangle boundary
        ax.plot([lon_min, lon_max], [lat_min, lat_min], 
                color=color, linewidth=linewidth, linestyle=linestyle, 
                alpha=alpha, transform=ccrs.PlateCarree())
        ax.plot([lon_min, lon_max], [lat_max, lat_max], 
                color=color, linewidth=linewidth, linestyle=linestyle,
                alpha=alpha, transform=ccrs.PlateCarree())
        ax.plot([lon_min, lon_min], [lat_min, lat_max], 
                color=color, linewidth=linewidth, linestyle=linestyle,
                alpha=alpha, transform=ccrs.PlateCarree())
        ax.plot([lon_max, lon_max], [lat_min, lat_max], 
                color=color, linewidth=linewidth, linestyle=linestyle,
                alpha=alpha, transform=ccrs.PlateCarree())
        
        if labels:
            center_lon = (lon_min + lon_max) / 2
            center_lat = (lat_min + lat_max) / 2
            ax.text(center_lon, center_lat, region_id,
                    transform=ccrs.PlateCarree(),
                    fontsize=6, fontweight='bold',
                    ha='center', va='center',
                    color='black', alpha=0.8,
                    bbox=dict(boxstyle='round,pad=0.2', 
                              facecolor='white', edgecolor='none', alpha=0.7))


In [ ]:
def plot_threshold_analysis(baseline_data, ssp585_data, ssp534os_data,
                            lat, lon, land_mask, threshold_label, output_dir, metrics_to_plot,
                            sig_ssp585=None, sig_ssp534os=None):
    """Plot analysis for a single threshold with proper land masking and significance stippling.
    
    Creates two separate figures:
    1. Temperature indices (Annual GDD, GDD Intensity, OGD, WSDI)
    2. Precipitation indices (R1mm, Rx5day, CWD, CDD, SDII, PRCPTOT)
    
    Parameters:
    -----------
    sig_ssp585 : dict, optional
        Significance masks for SSP5-8.5 - baseline {metric: sig_mask}
    sig_ssp534os : dict, optional
        Significance masks for SSP5-3.4-OS - baseline {metric: sig_mask}
    """
    # Define metric categories
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    # Filter to only metrics that are in metrics_to_plot
    temp_metrics_plot = [m for m in temp_metrics if m in metrics_to_plot]
    precip_metrics_plot = [m for m in precip_metrics if m in metrics_to_plot]
    
    def plot_metric_group(metrics_list, category_name, filename_suffix):
        """Plot a group of metrics."""
        n_metrics = len(metrics_list)
        if n_metrics == 0:
            return
        
        fig = plt.figure(figsize=(24, 6*n_metrics))
        # 6 columns: baseline, cbar, gap, ssp585_diff, ssp534os_diff, cbar
        gs = fig.add_gridspec(n_metrics, 6, width_ratios=[1, 0.015, 0.04, 1, 1, 0.015], 
                              wspace=0.02, hspace=0.15,
                              left=0.10, right=0.97, top=0.90, bottom=0.04)
        
        for mi, metric in enumerate(metrics_list):
            cfg = metrics_config.get(metric, {'label': metric, 'units': '', 'cmap_type': 'temp_increase'})
            diff_cmap = get_diff_cmap(cfg['cmap_type'])
            baseline_cmap = get_baseline_cmap(cfg['cmap_type'])
            
            baseline_metric = baseline_data.get(metric) if baseline_data else None
            ssp585_metric = ssp585_data.get(metric) if ssp585_data else None
            ssp534os_metric = ssp534os_data.get(metric) if ssp534os_data else None
            
            if baseline_metric is None:
                continue
            
            # Squeeze any extra dimensions from xclim output
            baseline_metric = np.squeeze(baseline_metric)
            if ssp585_metric is not None:
                ssp585_metric = np.squeeze(ssp585_metric)
            if ssp534os_metric is not None:
                ssp534os_metric = np.squeeze(ssp534os_metric)
            
            # Calculate color limits using land-masked data
            abs_data = baseline_metric[land_mask]
            vmin_abs = np.nanpercentile(abs_data, 2)
            vmax_abs = np.nanpercentile(abs_data, 98)
            
            diff_vals = []
            if ssp585_metric is not None:
                diff_vals.append((ssp585_metric - baseline_metric)[land_mask])
            if ssp534os_metric is not None:
                diff_vals.append((ssp534os_metric - baseline_metric)[land_mask])
            
            vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_vals)), 95) if diff_vals else 1
            
            im_abs, im_diff = None, None
            
            # Column 0: Baseline
            ax = fig.add_subplot(gs[mi, 0], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            pdata = baseline_metric.copy()
            pdata[~land_mask] = np.nan
            im_abs = ax.pcolormesh(lon, lat, pdata, transform=ccrs.PlateCarree(), 
                                   cmap=baseline_cmap, vmin=vmin_abs, vmax=vmax_abs)
            add_ipcc_boundaries(ax, labels=(mi == 0))  # Labels only on first row
            if mi == 0:
                ax.set_title(f'{threshold_label} Baseline', fontsize=11, fontweight='bold')
            ax.text(-0.18, 0.5, cfg['label'], transform=ax.transAxes, fontsize=14, 
                    fontweight='bold', rotation=90, va='center', ha='center')
            
            # Column 1: Baseline colorbar
            cax = fig.add_subplot(gs[mi, 1])
            cb = plt.colorbar(im_abs, cax=cax, extend='both', extendfrac=0.05)
            cb.set_label(cfg['units'], fontsize=13, fontweight='bold')
            cb.ax.tick_params(labelsize=11)
            
            # Column 2 is gap (no subplot)
            
            # Column 3: SSP5-8.5 - Baseline
            ax = fig.add_subplot(gs[mi, 3], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            if ssp585_metric is not None:
                diff = ssp585_metric - baseline_metric
                diff[~land_mask] = np.nan
                im_diff = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(), 
                                        cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff)
                # Add significance stippling
                if sig_ssp585 is not None and metric in sig_ssp585:
                    add_stippling(ax, lon, lat, sig_ssp585[metric], land_mask)
            add_ipcc_boundaries(ax, labels=False)
            if mi == 0:
                ax.set_title('SSP5-8.5\n- Baseline', fontsize=11, fontweight='bold')
            
            # Column 4: SSP5-3.4-OS - Baseline
            ax = fig.add_subplot(gs[mi, 4], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            if ssp534os_metric is not None:
                diff = ssp534os_metric - baseline_metric
                diff[~land_mask] = np.nan
                im_diff = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(), 
                                        cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff)
                # Add significance stippling
                if sig_ssp534os is not None and metric in sig_ssp534os:
                    add_stippling(ax, lon, lat, sig_ssp534os[metric], land_mask)
            add_ipcc_boundaries(ax, labels=False)
            if mi == 0:
                ax.set_title('SSP5-3.4-OS\n- Baseline', fontsize=11, fontweight='bold')
            
            # Column 5: Difference colorbar
            if im_diff is not None:
                cax = fig.add_subplot(gs[mi, 5])
                cb = plt.colorbar(im_diff, cax=cax, extend='both', extendfrac=0.05)
                cb.set_label(f'Delta {cfg["units"]}', fontsize=13, fontweight='bold')
                cb.ax.tick_params(labelsize=11)
        
        fig.suptitle(f'Climate Analysis - {threshold_label} Threshold - {category_name}\n(Overshoot 2060-2079 - Baseline, stippling: p<0.05, dashed lines: IPCC regions)', 
                     fontsize=14, fontweight='bold')
        
        filename = f'analysis_{threshold_label.replace(".", "")}_{filename_suffix}.png'
        plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"  Saved: {filename}")
    
    # Plot temperature indices
    if temp_metrics_plot:
        plot_metric_group(temp_metrics_plot, 'Temperature Indices', 'temp')
    
    # Plot precipitation indices
    if precip_metrics_plot:
        plot_metric_group(precip_metrics_plot, 'Precipitation Indices', 'precip')


In [ ]:
def plot_split_threshold_analysis(period_means_15C, period_means_20C, lat, lon, land_mask, output_dir, metrics_to_plot,
                                  sig_15C=None, sig_20C=None):
    """Plot 1.5C and 2.0C threshold analysis as SEPARATE plots.
    
    Creates 4 separate figures:
    - Temperature indices 1.5°C
    - Temperature indices 2.0°C
    - Precipitation indices 1.5°C
    - Precipitation indices 2.0°C
    """
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    temp_metrics_plot = [m for m in temp_metrics if m in metrics_to_plot]
    precip_metrics_plot = [m for m in precip_metrics if m in metrics_to_plot]
    
    def plot_single_threshold(metrics_list, category_name, threshold_label, period_means, sig_data, filename_suffix):
        """Plot a single threshold (1.5C or 2.0C) for a metric category."""
        n_metrics = len(metrics_list)
        if n_metrics == 0:
            return
        
        # Layout: 5 columns - Baseline | cbar | SSP5-8.5 diff | SSP5-3.4-OS diff | cbar
        fig = plt.figure(figsize=(22, 5*n_metrics))
        gs = fig.add_gridspec(n_metrics, 5, width_ratios=[1, 0.03, 1, 1, 0.03],
                              wspace=0.05, hspace=0.15,
                              left=0.08, right=0.97, top=0.92, bottom=0.03)
        
        for mi, metric in enumerate(metrics_list):
            cfg = metrics_config.get(metric, {'label': metric, 'units': '', 'cmap_type': 'temp_increase'})
            diff_cmap = get_diff_cmap(cfg['cmap_type'])
            baseline_cmap = get_baseline_cmap(cfg['cmap_type'])
            
            baseline = period_means['baseline'].get(metric)
            ssp585 = period_means['overshoot'].get('ssp585', {}).get(metric) if isinstance(period_means['overshoot'], dict) else period_means['overshoot'].get(metric)
            ssp534os = period_means['overshoot'].get('full_ssp534OS', {}).get(metric) if isinstance(period_means['overshoot'], dict) else None
            
            if baseline is None:
                continue
            
            baseline = np.squeeze(baseline)
            if ssp585 is not None: ssp585 = np.squeeze(ssp585)
            if ssp534os is not None: ssp534os = np.squeeze(ssp534os)
            
            vmin_abs = np.nanpercentile(baseline[land_mask], 2)
            vmax_abs = np.nanpercentile(baseline[land_mask], 98)
            
            diff_vals = []
            if ssp585 is not None: diff_vals.append((ssp585 - baseline)[land_mask])
            if ssp534os is not None: diff_vals.append((ssp534os - baseline)[land_mask])
            vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_vals)), 95) if diff_vals else 1
            
            im_abs, im_diff = None, None
            
            # Column 0: Baseline
            ax = fig.add_subplot(gs[mi, 0], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            pdata = baseline.copy()
            pdata[~land_mask] = np.nan
            im_abs = ax.pcolormesh(lon, lat, pdata, transform=ccrs.PlateCarree(),
                                   cmap=baseline_cmap, vmin=vmin_abs, vmax=vmax_abs)
            add_ipcc_boundaries(ax, labels=(mi == 0))
            if mi == 0: ax.set_title(f'{threshold_label} Baseline', fontsize=14, fontweight='bold')
            label_text = cfg['label']
            if len(label_text.split()) > 2:
                words = label_text.split()
                label_text = ' '.join(words[:2]) + '\n' + ' '.join(words[2:])
            ax.text(-0.15, 0.5, label_text, transform=ax.transAxes, fontsize=13,
                    fontweight='bold', rotation=90, va='center', ha='center')
            
            # Column 1: Baseline colorbar
            cax = fig.add_subplot(gs[mi, 1])
            cb = plt.colorbar(im_abs, cax=cax, extend='both')
            cb.ax.tick_params(labelsize=11)
            cb.set_label(cfg['units'], fontsize=12, fontweight='bold')
            
            # Column 2: SSP5-8.5 - Baseline
            ax = fig.add_subplot(gs[mi, 2], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            if ssp585 is not None:
                diff = ssp585 - baseline
                diff[~land_mask] = np.nan
                im_diff = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(),
                                        cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff)
                if sig_data and 'ssp585' in sig_data and metric in sig_data['ssp585']:
                    add_stippling(ax, lon, lat, sig_data['ssp585'][metric], land_mask)
            add_ipcc_boundaries(ax, labels=False)
            if mi == 0: ax.set_title(f'{threshold_label} SSP5-8.5\n- Baseline', fontsize=14, fontweight='bold')
            
            # Column 3: SSP5-3.4-OS - Baseline
            ax = fig.add_subplot(gs[mi, 3], projection=ccrs.PlateCarree())
            setup_africa_map(ax)
            if ssp534os is not None:
                diff = ssp534os - baseline
                diff[~land_mask] = np.nan
                im_diff = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(),
                                        cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff)
                if sig_data and 'ssp534os' in sig_data and metric in sig_data['ssp534os']:
                    add_stippling(ax, lon, lat, sig_data['ssp534os'][metric], land_mask)
            add_ipcc_boundaries(ax, labels=False)
            if mi == 0: ax.set_title(f'{threshold_label} SSP5-3.4-OS\n- Baseline', fontsize=14, fontweight='bold')
            
            # Column 4: Diff colorbar
            if im_diff is not None:
                cax = fig.add_subplot(gs[mi, 4])
                cb = plt.colorbar(im_diff, cax=cax, extend='both')
                cb.ax.tick_params(labelsize=11)
                cb.set_label(f'Δ {cfg["units"]}', fontsize=12, fontweight='bold')
        
        fig.suptitle(f'{category_name} - {threshold_label} Threshold\n(Overshoot 2060-2079 - Baseline, stippling: p<0.05)',
                     fontsize=18, fontweight='bold')
        
        filename = f'analysis_{filename_suffix}_{threshold_label.replace(".", "").replace("°", "")}.png'
        plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"  Saved: {filename}")
    
    # Create 4 separate plots
    if temp_metrics_plot:
        plot_single_threshold(temp_metrics_plot, 'Temperature Indices', '1.5°C', period_means_15C, sig_15C, 'temp')
        plot_single_threshold(temp_metrics_plot, 'Temperature Indices', '2.0°C', period_means_20C, sig_20C, 'temp')
    if precip_metrics_plot:
        plot_single_threshold(precip_metrics_plot, 'Precipitation Indices', '1.5°C', period_means_15C, sig_15C, 'precip')
        plot_single_threshold(precip_metrics_plot, 'Precipitation Indices', '2.0°C', period_means_20C, sig_20C, 'precip')

# Keep old function name for compatibility
plot_combined_threshold_analysis = plot_split_threshold_analysis


In [ ]:
def plot_sai_impact(period_means_15C, period_means_20C, lat, lon, land_mask, output_dir, metrics_to_plot):
    """Plot SAI impact with proper land masking.
    
    Creates two separate figures:
    1. Temperature indices (Annual GDD, GDD Intensity, OGD, WSDI)
    2. Precipitation indices (R1mm, Rx5day, CWD, CDD, SDII, PRCPTOT)
    """
    # Define metric categories
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    # Filter to only metrics that are in metrics_to_plot
    temp_metrics_plot = [m for m in temp_metrics if m in metrics_to_plot]
    precip_metrics_plot = [m for m in precip_metrics if m in metrics_to_plot]
    
    comparisons = [
        ('full_ssp585_15_feedback', 'ssp585', 'SAI 1.5°C\n- SSP5-8.5', '15C'),
        ('full_ssp534OS_15_feedback', 'full_ssp534OS', 'SAI 1.5°C\n- SSP5-3.4-OS', '15C'),
        ('full_ssp534OS_20_feedback', 'ssp585', 'SAI 2.0°C\n- SSP5-8.5', '20C'),
        ('full_ssp534OS_20_feedback', 'full_ssp534OS', 'SAI 2.0°C\n- SSP5-3.4-OS', '20C')
    ]
    
    def plot_metric_group(metrics_list, category_name, filename_suffix):
        """Plot SAI impact for a group of metrics."""
        n_metrics = len(metrics_list)
        if n_metrics == 0:
            return
        
        fig = plt.figure(figsize=(28, 5*n_metrics))
        gs = fig.add_gridspec(n_metrics, 5, width_ratios=[1, 1, 1, 1, 0.03], 
                              wspace=0.02, hspace=0.12,
                              left=0.08, right=0.95, top=0.90, bottom=0.04)
        
        for mi, metric in enumerate(metrics_list):
            cfg = metrics_config.get(metric, {'label': metric, 'units': '', 'cmap_type': 'temp_increase'})
            diff_cmap = get_diff_cmap(cfg['cmap_type'])
            
            # Calculate vmax from land-masked data
            all_diffs = []
            for sai_scen, ref_scen, _, thresh in comparisons:
                pm = period_means_15C if thresh == '15C' else period_means_20C
                if sai_scen in pm and ref_scen in pm:
                    if metric in pm[sai_scen] and metric in pm[ref_scen]:
                        sai_data = np.squeeze(pm[sai_scen][metric])
                        ref_data = np.squeeze(pm[ref_scen][metric])
                        diff = sai_data - ref_data
                        all_diffs.append(diff[land_mask])
            
            vmax = np.nanpercentile(np.abs(np.concatenate(all_diffs)), 95) if all_diffs else 1
            im = None
            
            for ci, (sai_scen, ref_scen, title, thresh) in enumerate(comparisons):
                ax = fig.add_subplot(gs[mi, ci], projection=ccrs.PlateCarree())
                setup_africa_map(ax)
                
                pm = period_means_15C if thresh == '15C' else period_means_20C
                if sai_scen in pm and ref_scen in pm:
                    if metric in pm[sai_scen] and metric in pm[ref_scen]:
                        sai_data = np.squeeze(pm[sai_scen][metric])
                        ref_data = np.squeeze(pm[ref_scen][metric])
                        diff = sai_data - ref_data
                        diff[~land_mask] = np.nan
                        im = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(), 
                                           cmap=diff_cmap, vmin=-vmax, vmax=vmax)
                
                # Add IPCC boundaries (labels only on first column of first row)
                add_ipcc_boundaries(ax, labels=(mi == 0 and ci == 0))
                
                if mi == 0:
                    ax.set_title(title, fontsize=10, fontweight='bold')
                if ci == 0:
                    ax.text(-0.15, 0.5, cfg['label'], transform=ax.transAxes, fontsize=14, 
                            fontweight='bold', rotation=90, va='center', ha='center')
            
            if im is not None:
                cax = fig.add_subplot(gs[mi, 4])
                cb = plt.colorbar(im, cax=cax, extend='both', extendfrac=0.05)
                cb.set_label(f'Delta {cfg["units"]}', fontsize=13, fontweight='bold')
                cb.ax.tick_params(labelsize=11)
        
        fig.suptitle(f'SAI Impact Analysis - {category_name}\n(Overshoot Period 2060-2079)', fontsize=18, fontweight='bold')
        filename = f'sai_impact_{filename_suffix}.png'
        plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"  Saved: {filename}")
    
    # Plot temperature indices
    if temp_metrics_plot:
        plot_metric_group(temp_metrics_plot, 'Temperature Indices', 'temp')
    
    # Plot precipitation indices
    if precip_metrics_plot:
        plot_metric_group(precip_metrics_plot, 'Precipitation Indices', 'precip')


In [ ]:
def plot_temporal(all_results, output_dir, metrics_to_plot):
    """Create temporal analysis plots.
    
    Creates two separate figures:
    1. Temperature indices (Annual GDD, GDD Intensity, OGD, WSDI)
    2. Precipitation indices (R1mm, Rx5day, CWD, CDD, SDII, PRCPTOT)
    """
    # Define metric categories
    temp_metrics = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi']
    precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
    
    # Filter to only metrics that are in metrics_to_plot
    temp_metrics_plot = [m for m in temp_metrics if m in metrics_to_plot]
    precip_metrics_plot = [m for m in precip_metrics if m in metrics_to_plot]
    
    scenarios = [
        ('full_ssp585_15_feedback', 'SSP5-8.5 1.5 feedback', '#1f77b4', '--'),
        ('full_ssp534OS_15_feedback', 'SSP5-3.4OS 1.5 feedback', '#2ca02c', '--'),
        ('full_ssp534OS_20_feedback', 'SSP5-3.4OS 2.0 feedback', '#ff7f0e', '--'),
        ('full_ssp534OS', 'SSP5-3.4OS', '#d62728', '-'),
        ('ssp585', 'SSP5-8.5', '#000000', '-')
    ]
    
    def plot_metric_group(metrics_list, category_name, filename_suffix):
        """Plot temporal analysis for a group of metrics."""
        n_metrics = len(metrics_list)
        if n_metrics == 0:
            return
        
        n_cols = 2
        n_rows = (n_metrics + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5*n_rows))
        fig.patch.set_facecolor('white')
        axes = axes.flatten() if n_metrics > 1 else [axes]
        
        for idx, metric in enumerate(metrics_list):
            if idx >= len(axes): break
            ax = axes[idx]
            ax.set_facecolor('white')
            cfg = metrics_config.get(metric, {'label': metric, 'units': ''})
            
            for sname, label, color, ls in scenarios:
                if sname not in all_results: continue
                
                all_years, all_vals = None, []
                for ens, ens_results in all_results[sname].items():
                    if metric not in ens_results: continue
                    if ens_results[metric] is None: continue
                    yrs = get_years_for_metric(ens_results, metric)
                    if yrs is None: continue
                    yrs = yrs.astype(int)
                    metric_data = np.squeeze(ens_results[metric])
                    vals = np.nanmean(metric_data, axis=(1, 2))
                    if len(vals) != len(yrs): continue
                    mask = yrs <= 2099
                    if np.sum(mask) > 0:
                        if all_years is None: all_years = yrs[mask]
                        all_vals.append(vals[mask])
                
                if all_vals:
                    minlen = min(len(v) for v in all_vals)
                    arr = np.array([v[:minlen] for v in all_vals])
                    mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                    ax.plot(all_years[:minlen], mean, color=color, linewidth=2.5, label=label, linestyle=ls)
                    if len(arr) > 1:
                        std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                        ax.fill_between(all_years[:minlen], mean-std, mean+std, color=color, alpha=0.15)
            
            # Add period shading
            ax.axvspan(2015, 2034, color='dodgerblue', alpha=0.15)
            ax.axvspan(2024, 2044, color='orange', alpha=0.15)
            ax.axvspan(2060, 2079, color='red', alpha=0.1)
            ax.axvspan(2080, 2099, color='blue', alpha=0.1)
            ax.axvline(x=2040, color='black', linestyle=':', linewidth=2, alpha=0.7)
            ax.axvline(x=2080, color='purple', linestyle=':', linewidth=2, alpha=0.7)
            
            # Add period annotations (only on first subplot)
            if idx == 0:
                ylim = ax.get_ylim()
                yrange = ylim[1] - ylim[0]
                ax.text(2024.5, ylim[0] + 0.35*yrange, '1.5C', ha='center', fontsize=8, color='dodgerblue',
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                ax.text(2034, ylim[0] + 0.25*yrange, '2.0C', ha='center', fontsize=8, color='orange',
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                ax.text(2070, ylim[1] - 0.08*yrange, 'Overshoot', ha='center', fontsize=9, color='red',
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                ax.text(2090, ylim[1] - 0.08*yrange, 'Recovery', ha='center', fontsize=9, color='blue',
                       fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            
            ax.set_xlabel('Year')
            ax.set_ylabel(f"{cfg['label']} ({cfg['units']})")
            ax.set_title(cfg['label'], fontweight='bold')
            if idx == 0:
                ax.legend(fontsize=7, loc='upper left')
            ax.set_xlim(2015, 2100)
            ax.grid(True, alpha=0.3, linestyle='--')
        
        for idx in range(n_metrics, len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(f'{category_name}: Overshoot to Recovery (2015-2099)', fontsize=15, fontweight='bold')
        plt.tight_layout(rect=[0, 0.02, 1, 0.96])
        filename = f'temporal_{filename_suffix}.png'
        plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"  Saved: {filename}")
    
    # Plot temperature indices
    if temp_metrics_plot:
        plot_metric_group(temp_metrics_plot, 'Temperature Indices', 'temp')
    
    # Plot precipitation indices
    if precip_metrics_plot:
        plot_metric_group(precip_metrics_plot, 'Precipitation Indices', 'precip')


In [ ]:
def compute_seasonal_data(temp_files, precip_files, scenarios_to_process):
    """Compute seasonal means for temperature and precipitation.
    
    Seasons:
    - DJF (December-January-February): Dry season in many African regions
    - MAM (March-April-May): Long rains (East Africa)
    - JJA (June-July-August): West African monsoon
    - SON (September-October-November): Short rains (East Africa)
    """
    print("\nComputing seasonal data...")
    
    seasonal_data = {}
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    
    for scenario in scenarios_to_process:
        # Get files for this scenario (files are dicts with 'path', 'scenario', 'ensemble')
        temp_scenario_files = [f for f in temp_files if f['scenario'] == scenario]
        precip_scenario_files = [f for f in precip_files if f['scenario'] == scenario]
        
        if not temp_scenario_files and not precip_scenario_files:
            continue
        
        seasonal_data[scenario] = {}
        
        # Process temperature files
        for finfo in temp_scenario_files[:3]:  # Limit to 3 ensemble members
            filepath = finfo['path']
            ens = finfo['ensemble']
            if ens not in seasonal_data[scenario]:
                seasonal_data[scenario][ens] = {}
            
            try:
                ds = xr.open_dataset(filepath, decode_times=False)
                ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                            lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                ds = xr.decode_cf(ds)
                ds = remove_duplicate_times(ds)
                
                if 'TREFHT' in ds.data_vars:
                    # Convert to Celsius
                    tas = ds['TREFHT'] - 273.15
                    
                    # Compute seasonal means
                    seasonal_temp = tas.resample(time='QS-DEC').mean(dim='time')
                    
                    # Extract by season
                    for season in seasons:
                        if season == 'DJF':
                            season_data = seasonal_temp.sel(time=seasonal_temp.time.dt.month == 12)
                        elif season == 'MAM':
                            season_data = seasonal_temp.sel(time=seasonal_temp.time.dt.month == 3)
                        elif season == 'JJA':
                            season_data = seasonal_temp.sel(time=seasonal_temp.time.dt.month == 6)
                        else:  # SON
                            season_data = seasonal_temp.sel(time=seasonal_temp.time.dt.month == 9)
                        
                        seasonal_data[scenario][ens][f'temp_{season}'] = season_data.values
                    
                    seasonal_data[scenario][ens]['years_seasonal'] = seasonal_temp.time.dt.year.values[::4][:len(seasonal_temp)//4]
                    seasonal_data[scenario][ens]['lat'] = ds.lat.values
                    seasonal_data[scenario][ens]['lon'] = ds.lon.values
                
                ds.close()
            except Exception as e:
                print(f"  Error processing {filepath}: {e}")
        
        # Process precipitation files
        for finfo in precip_scenario_files[:3]:
            filepath = finfo['path']
            ens = finfo['ensemble']
            if ens not in seasonal_data[scenario]:
                seasonal_data[scenario][ens] = {}
            
            try:
                ds = xr.open_dataset(filepath, decode_times=False)
                ds = ds.sel(lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                            lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                ds = xr.decode_cf(ds)
                ds = remove_duplicate_times(ds)
                
                if 'PRECT' in ds.data_vars:
                    # Convert to mm/day
                    pr = ds['PRECT'] * 86400 * 1000
                    
                    # Compute seasonal totals
                    seasonal_precip = pr.resample(time='QS-DEC').sum(dim='time')
                    
                    # Extract by season
                    for season in seasons:
                        if season == 'DJF':
                            season_data = seasonal_precip.sel(time=seasonal_precip.time.dt.month == 12)
                        elif season == 'MAM':
                            season_data = seasonal_precip.sel(time=seasonal_precip.time.dt.month == 3)
                        elif season == 'JJA':
                            season_data = seasonal_precip.sel(time=seasonal_precip.time.dt.month == 6)
                        else:  # SON
                            season_data = seasonal_precip.sel(time=seasonal_precip.time.dt.month == 9)
                        
                        seasonal_data[scenario][ens][f'precip_{season}'] = season_data.values
                    
                    if 'years_seasonal' not in seasonal_data[scenario][ens]:
                        seasonal_data[scenario][ens]['years_seasonal'] = seasonal_precip.time.dt.year.values[::4][:len(seasonal_precip)//4]
                    if 'lat' not in seasonal_data[scenario][ens]:
                        seasonal_data[scenario][ens]['lat'] = ds.lat.values
                        seasonal_data[scenario][ens]['lon'] = ds.lon.values
                
                ds.close()
            except Exception as e:
                print(f"  Error processing {filepath}: {e}")
        
        print(f"  Processed seasonal data for {scenario}")
    
    return seasonal_data

def plot_seasonal_analysis(seasonal_data, output_dir):
    """Plot seasonal analysis for temperature and precipitation."""
    print("\nPlotting seasonal analysis...")
    
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    season_names = {'DJF': 'Dec-Jan-Feb', 'MAM': 'Mar-Apr-May', 
                    'JJA': 'Jun-Jul-Aug', 'SON': 'Sep-Oct-Nov'}
    
    # Use file scenario names (without 'full_' prefix)
    scenarios_plot = [
        ('ssp585_15_feedback', 'SSP5-8.5 1.5°C SAI', '#1f77b4', '--'),
        ('ssp534OS_15_feedback', 'SSP5-3.4OS 1.5°C SAI', '#2ca02c', '--'),
        ('ssp534OS_20_feedback', 'SSP5-3.4OS 2.0°C SAI', '#ff7f0e', '--'),
        ('ssp534OS', 'SSP5-3.4OS', '#d62728', '-'),
        ('ssp585', 'SSP5-8.5', '#000000', '-')
    ]
    
    # Temperature seasonal plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('white')
    axes = axes.flatten()
    
    for idx, season in enumerate(seasons):
        ax = axes[idx]
        ax.set_facecolor('white')
        
        for sname, label, color, ls in scenarios_plot:
            if sname not in seasonal_data:
                continue
            
            all_years, all_vals = None, []
            for ens, ens_data in seasonal_data[sname].items():
                metric_key = f'temp_{season}'
                if metric_key not in ens_data:
                    continue
                
                yrs = ens_data.get('years_seasonal')
                if yrs is None:
                    continue
                
                vals = np.nanmean(ens_data[metric_key], axis=(1, 2))
                if len(vals) > len(yrs):
                    vals = vals[:len(yrs)]
                
                mask = yrs <= 2099
                if np.sum(mask) > 0:
                    if all_years is None:
                        all_years = yrs[mask]
                    all_vals.append(vals[mask])
            
            if all_vals:
                minlen = min(len(v) for v in all_vals)
                arr = np.array([v[:minlen] for v in all_vals])
                mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
                if len(arr) > 1:
                    std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                    ax.fill_between(all_years[:minlen], mean-std, mean+std, color=color, alpha=0.15)
        
        ax.set_xlabel('Year')
        ax.set_ylabel('Temperature (°C)')
        ax.set_title(f'{season_names[season]} ({season})', fontweight='bold')
        ax.set_xlim(2015, 2100)
        ax.grid(True, alpha=0.3, linestyle='--')
        if idx == 0:
            ax.legend(fontsize=7, loc='upper left')
    
    plt.suptitle('Seasonal Temperature Analysis - Africa', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{output_dir}/seasonal_temperature.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: seasonal_temperature.png")
    
    # Precipitation seasonal plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('white')
    axes = axes.flatten()
    
    for idx, season in enumerate(seasons):
        ax = axes[idx]
        ax.set_facecolor('white')
        
        for sname, label, color, ls in scenarios_plot:
            if sname not in seasonal_data:
                continue
            
            all_years, all_vals = None, []
            for ens, ens_data in seasonal_data[sname].items():
                metric_key = f'precip_{season}'
                if metric_key not in ens_data:
                    continue
                
                yrs = ens_data.get('years_seasonal')
                if yrs is None:
                    continue
                
                vals = np.nanmean(ens_data[metric_key], axis=(1, 2))
                if len(vals) > len(yrs):
                    vals = vals[:len(yrs)]
                
                mask = yrs <= 2099
                if np.sum(mask) > 0:
                    if all_years is None:
                        all_years = yrs[mask]
                    all_vals.append(vals[mask])
            
            if all_vals:
                minlen = min(len(v) for v in all_vals)
                arr = np.array([v[:minlen] for v in all_vals])
                mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
                if len(arr) > 1:
                    std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                    ax.fill_between(all_years[:minlen], mean-std, mean+std, color=color, alpha=0.15)
        
        ax.set_xlabel('Year')
        ax.set_ylabel('Precipitation (mm)')
        ax.set_title(f'{season_names[season]} ({season})', fontweight='bold')
        ax.set_xlim(2015, 2100)
        ax.grid(True, alpha=0.3, linestyle='--')
        if idx == 0:
            ax.legend(fontsize=7, loc='upper left')
    
    plt.suptitle('Seasonal Precipitation Analysis - Africa', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'{output_dir}/seasonal_precipitation.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: seasonal_precipitation.png")


## Main Execution


In [ ]:
# Scan for data files

temp_files_raw = glob.glob(f"{temp_data_dir}/*TREFHT*.nc")
temp_files_raw = [f for f in temp_files_raw if 'TREFHTMX' not in f and 'TREFHTMN' not in f]
temp_files = get_file_info(temp_files_raw)
print(f"  TREFHT files: {len(temp_files)}")

tmax_files_raw = glob.glob(f"{tmax_data_dir}/*TREFHTMX*.nc")
tmax_files = get_file_info(tmax_files_raw)
print(f"  TREFHTMX files: {len(tmax_files)}")

prec_files_raw = glob.glob(f"{prec_data_dir}/*PRECT*.nc")
prec_files = get_file_info(prec_files_raw)
print(f"  PRECT files: {len(prec_files)}")

# Show scenarios found
all_scenarios = set([f['scenario'] for f in temp_files + tmax_files + prec_files])
print(f"\nScenarios found: {sorted(all_scenarios)}")


In [ ]:
# Compute baseline thresholds for WSDI using xclim
tasmax_per = compute_baseline_percentile(tmax_files, baseline_period=(2015, 2034))


In [ ]:
# Process all data using xclim
print("Ensemble averaging happens in compute_period_means().")

all_results = process_all_data(temp_files, tmax_files, prec_files, tasmax_per)

print(f"\nProcessed {len(all_results)} scenarios")
for scen in sorted(all_results.keys()):
    print(f"  {scen}: {len(all_results[scen])} ensemble(s)")


In [ ]:
# Merge scenarios

merge_scenario_results(all_results, 'ssp585', 'ssp534OS', 'full_ssp534OS', 2040)
merge_scenario_results(all_results, 'ssp585', 'ssp534OS_15_feedback', 'full_ssp534OS_15_feedback', 2040)
merge_scenario_results(all_results, 'ssp585', 'ssp534OS_20_feedback', 'full_ssp534OS_20_feedback', 2040)
merge_scenario_results(all_results, 'ssp585', 'ssp585_15_feedback', 'full_ssp585_15_feedback', 2020)

print(f"\nFinal scenarios: {sorted(all_results.keys())}")


In [ ]:
# Save intermediate results
with open('all_results_xclim.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print("  Saved: all_results_xclim.pkl")


In [ ]:
# Compute period means (this is where ensemble averaging happens)

period_means_15C = {
    'baseline': compute_period_means(all_results, (2015, 2034)),
    'overshoot': compute_period_means(all_results, (2060, 2079))
}

period_means_20C = {
    'baseline': compute_period_means(all_results, (2024, 2044)),
    'overshoot': compute_period_means(all_results, (2060, 2079))
}

print("Period means computed")


In [ ]:
# Get coordinates and create land mask
lat, lon = None, None
for scen in all_results:
    for ens in all_results[scen]:
        if 'lat' in all_results[scen][ens] and 'lon' in all_results[scen][ens]:
            lat = all_results[scen][ens]['lat']
            lon = all_results[scen][ens]['lon']
            break
    if lat is not None:
        break

print(f"Grid: {len(lat)} lat x {len(lon)} lon")
land_mask = create_land_mask(lat, lon)


In [ ]:
# Create all plots

metrics_to_plot = ['mean_temperature', 'annual_gdd', 'gdd_intensity', 'optimal_growing_days', 'wsdi', 'r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']

# Compute per-ensemble data for statistical testing
print("\nComputing ensemble data for significance testing...")
ensemble_baseline_15C = compute_ensemble_period_data(all_results, (2015, 2034))
ensemble_overshoot = compute_ensemble_period_data(all_results, (2060, 2079))
ensemble_baseline_20C = compute_ensemble_period_data(all_results, (2024, 2044))

# Compute significance for 1.5°C threshold
print("Computing significance maps (1.5°C threshold)...")
sig_15C = compute_all_significance(
    ensemble_baseline_15C, ensemble_overshoot,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')]
)

# Compute significance for 2.0°C threshold
print("Computing significance maps (2.0°C threshold)...")
sig_20C = compute_all_significance(
    ensemble_baseline_20C, ensemble_overshoot,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')]
)

# 1.5°C Threshold Analysis
print("\n1.5°C Threshold Analysis...")
plot_threshold_analysis(
    baseline_data=period_means_15C['baseline'].get('ssp585'),
    ssp585_data=period_means_15C['overshoot'].get('ssp585'),
    ssp534os_data=period_means_15C['overshoot'].get('full_ssp534OS'),
    lat=lat, lon=lon, land_mask=land_mask,
    threshold_label='1.5C', output_dir=output_dir,
    metrics_to_plot=metrics_to_plot,
    sig_ssp585=sig_15C.get('ssp585'),
    sig_ssp534os=sig_15C.get('full_ssp534OS')
)

# 2.0°C Threshold Analysis
print("\n2.0°C Threshold Analysis...")
plot_threshold_analysis(
    baseline_data=period_means_20C['baseline'].get('ssp585'),
    ssp585_data=period_means_20C['overshoot'].get('ssp585'),
    ssp534os_data=period_means_20C['overshoot'].get('full_ssp534OS'),
    lat=lat, lon=lon, land_mask=land_mask,
    threshold_label='2.0C', output_dir=output_dir,
    metrics_to_plot=metrics_to_plot,
    sig_ssp585=sig_20C.get('ssp585'),
    sig_ssp534os=sig_20C.get('full_ssp534OS')
)

# Combined 1.5°C and 2.0°C Threshold Analysis
print("\nCombined Threshold Analysis (1.5°C & 2.0C)...")
# Restructure data for combined plotting
period_means_15C_combined = {
    'baseline': period_means_15C['baseline'].get('ssp585', {}),
    'overshoot': period_means_15C['overshoot']
}
period_means_20C_combined = {
    'baseline': period_means_20C['baseline'].get('ssp585', {}),
    'overshoot': period_means_20C['overshoot']
}
sig_15C_combined = {'ssp585': sig_15C.get('ssp585', {}), 'ssp534os': sig_15C.get('full_ssp534OS', {})}
sig_20C_combined = {'ssp585': sig_20C.get('ssp585', {}), 'ssp534os': sig_20C.get('full_ssp534OS', {})}
plot_combined_threshold_analysis(
    period_means_15C_combined, period_means_20C_combined,
    lat, lon, land_mask, output_dir, metrics_to_plot,
    sig_15C=sig_15C_combined, sig_20C=sig_20C_combined
)

# SAI Impact
print("\nSAI Impact Analysis...")
plot_sai_impact(
    period_means_15C=period_means_15C['overshoot'],
    period_means_20C=period_means_20C['overshoot'],
    lat=lat, lon=lon, land_mask=land_mask,
    output_dir=output_dir,
    metrics_to_plot=metrics_to_plot
)

# Temporal Analysis
print("\nTemporal Analysis...")
plot_temporal(all_results, output_dir, metrics_to_plot)

# Seasonal Analysis
print("\nSeasonal Analysis...")
# Note: Use file scenario names (without 'full_' prefix)
scenarios_for_seasonal = ['ssp585', 'ssp534OS', 'ssp585_15_feedback', 'ssp534OS_15_feedback', 'ssp534OS_20_feedback']
seasonal_data = compute_seasonal_data(temp_files, prec_files, scenarios_for_seasonal)
plot_seasonal_analysis(seasonal_data, output_dir)

# Compute and display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS & VISUALIZATIONS")

# 1.5°C Statistics Visualization
print("\nCreating 1.5°C statistics visualizations...")
plot_statistics_visualization(
    period_means_15C['baseline'], period_means_15C['overshoot'],
    ensemble_baseline_15C, ensemble_overshoot,
    land_mask,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')],
    '1.5C', output_dir
)

# 2.0°C Statistics Visualization
print("\nCreating 2.0°C statistics visualizations...")
plot_statistics_visualization(
    period_means_20C['baseline'], period_means_20C['overshoot'],
    ensemble_baseline_20C, ensemble_overshoot,
    land_mask,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')],
    '2.0C', output_dir
)

# SAI Statistics Visualization (Non-SAI - SAI comparison)
print("\nCreating SAI impact statistics visualizations...")
plot_sai_statistics(
    period_means_15C['overshoot'],  # Using overshoot period means which includes SAI scenarios
    ensemble_overshoot,
    land_mask,
    output_dir
)

# Combined 1.5°C and 2.0°C Statistics Visualization
print("\nCreating combined threshold statistics visualizations...")
plot_combined_statistics(
    period_means_15C['baseline'], period_means_15C['overshoot'],
    period_means_20C['baseline'], period_means_20C['overshoot'],
    ensemble_baseline_15C, ensemble_overshoot,
    ensemble_baseline_20C, ensemble_overshoot,
    land_mask,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')],
    output_dir
)

# Compute summary statistics for CSV export
summary_15C_df = compute_summary_statistics(
    period_means_15C['baseline'], period_means_15C['overshoot'],
    ensemble_baseline_15C, ensemble_overshoot,
    land_mask,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')]
)
summary_20C_df = compute_summary_statistics(
    period_means_20C['baseline'], period_means_20C['overshoot'],
    ensemble_baseline_20C, ensemble_overshoot,
    land_mask,
    [('ssp585', 'ssp585'), ('ssp585', 'full_ssp534OS')]
)

# Save summary tables
print("\nSaving summary tables...")
summary_15C_df.to_csv(f'{output_dir}/summary_statistics_15C.csv', index=False)
summary_20C_df.to_csv(f'{output_dir}/summary_statistics_20C.csv', index=False)
print(f"Saved summary tables to {output_dir}")

print("\n" + "="*60)
print("ALL ANALYSIS COMPLETE!")


In [ ]:
# FIG 1: TIME SERIES (3 rows x 2 cols)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

paired_metrics = [
    ('mean_temperature', 'prcptot'),
    ('wsdi',             'cwd'),
    ('gdd_intensity',    'sdii'),
]

scenarios_ts = [
    ('full_ssp585_15_feedback',    'SSP5-8.5-1.5 SAI',    '#1f77b4', '--'),
    ('full_ssp534OS_15_feedback',  'SSP5-3.4-OS-1.5 SAI', '#2ca02c', '--'),
    ('full_ssp534OS',              'SSP5-3.4-OS',          '#d62728', '-'),
    ('ssp585',                     'SSP5-8.5',             '#000000', '-'),
]

def _make_timeseries_fig(year_start, tag):
    """Build a 3x2 time-series figure starting from year_start."""
    fig, axes = plt.subplots(3, 2, figsize=(18, 14))
    fig.patch.set_facecolor('white')

    for row, (met_left, met_right) in enumerate(paired_metrics):
        for col, metric in enumerate([met_left, met_right]):
            ax = axes[row, col]
            ax.set_facecolor('white')
            cfg = metrics_config.get(metric, {'label': metric, 'units': ''})

            for sname, label, color, ls in scenarios_ts:
                if sname not in all_results:
                    continue
                all_years, all_vals = None, []
                for ens, ens_results in all_results[sname].items():
                    if metric not in ens_results or ens_results[metric] is None:
                        continue
                    yrs = get_years_for_metric(ens_results, metric)
                    if yrs is None:
                        continue
                    yrs = yrs.astype(int)
                    data = np.squeeze(ens_results[metric])
                    vals = np.nanmean(data, axis=(1, 2))
                    if len(vals) != len(yrs):
                        continue
                    keep = (yrs >= year_start) & (yrs <= 2099)
                    if np.sum(keep) > 0:
                        if all_years is None:
                            all_years = yrs[keep]
                        all_vals.append(vals[keep])

                if all_vals:
                    minlen = min(len(v) for v in all_vals)
                    arr = np.array([v[:minlen] for v in all_vals])
                    mean = pd.Series(np.nanmean(arr, axis=0)).rolling(
                        5, center=True, min_periods=1).mean().values
                    ax.plot(all_years[:minlen], mean, color=color,
                            linewidth=2.5, label=label, linestyle=ls)
                    if len(arr) > 1:
                        std = pd.Series(np.nanstd(arr, axis=0)).rolling(
                            5, center=True, min_periods=1).mean().values
                        ax.fill_between(all_years[:minlen],
                                        mean - std, mean + std,
                                        color=color, alpha=0.15)

            # Period shading
            for s, e, c, a in [(2015,2034,'dodgerblue',0.15),
                                (2024,2044,'orange',0.15),
                                (2060,2079,'red',0.10),
                                (2080,2099,'blue',0.10)]:
                if s >= year_start:
                    ax.axvspan(s, e, color=c, alpha=a)
            for xv, c in [(2040,'black'),(2080,'purple')]:
                if xv >= year_start:
                    ax.axvline(x=xv, color=c, linestyle=':', linewidth=2, alpha=0.7)

            # Period annotations on first subplot only
            if row == 0 and col == 0:
                ylim = ax.get_ylim()
                yrange = ylim[1] - ylim[0]
                if year_start <= 2034:
                    ax.text(2024.5, ylim[0] + 0.35*yrange, '1.5C',
                            ha='center', fontsize=8, color='dodgerblue', fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                    ax.text(2034, ylim[0] + 0.25*yrange, '2.0C',
                            ha='center', fontsize=8, color='orange', fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                ax.text(2070, ylim[1] - 0.08*yrange, 'Overshoot',
                        ha='center', fontsize=9, color='red', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
                ax.text(2090, ylim[1] - 0.08*yrange, 'Recovery',
                        ha='center', fontsize=9, color='blue', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

            ax.set_ylabel(f"{cfg['label']} ({cfg['units']})", fontsize=12, fontweight='bold')
            ax.set_title(cfg['label'], fontsize=13, fontweight='bold')
            ax.set_xlim(year_start, 2100)
            ax.grid(True, alpha=0.3, linestyle='--')
            ax.tick_params(labelsize=11)
            # Legend: top-left of first subplot only
            if row == 0 and col == 0:
                ax.legend(fontsize=7, loc='upper left')
            if row == 2:
                ax.set_xlabel('Year', fontsize=12, fontweight='bold')

    fig.suptitle(
        f'Climate Indices: Overshoot to Recovery ({year_start}\u20132099)',
        fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0.02, 1, 0.96])
    fname = f'{output_dir}/fig1_timeseries_{year_start}_2099.png'
    plt.savefig(fname, dpi=400, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f'Saved: {fname}')

_make_timeseries_fig(2015, 'full')
_make_timeseries_fig(2040, 'from2040')
print('Fig 1 complete.')


In [ ]:
# FIG 2: SPATIAL MAPS (4 rows x 5 cols, hatch = not significant)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Patch

fig2_metrics = ['mean_temperature', 'prcptot', 'gdd_intensity', 'cwd']

columns = [
    (None,                          'Baseline\n(2015\u20132034)',           True),
    ('ssp585',                      'SSP5-8.5',                       False),
    ('full_ssp534OS',               'SSP5-3.4-OS',                    False),
    ('full_ssp585_15_feedback',     'SSP5-8.5-1.5 SAI',              False),
    ('full_ssp534OS_15_feedback',   'SSP5-3.4-OS-1.5 SAI',           False),
]

baseline_data_fig2  = period_means_15C['baseline'].get('ssp585', {})
overshoot_data_fig2 = period_means_15C['overshoot']

def add_hatching_nonsig(ax, lon, lat, sig_mask, land_mask):
    """Hatch land areas that are NOT statistically significant (p >= 0.05)."""
    if sig_mask is None:
        return
    nonsig = (~sig_mask) & land_mask
    lon2d, lat2d = np.meshgrid(lon, lat)
    ax.contourf(lon2d, lat2d, nonsig.astype(float),
                levels=[0.5, 1.5], colors='none',
                hatches=['////'], transform=ccrs.PlateCarree())

n_rows = len(fig2_metrics)
n_cols = len(columns)

fig = plt.figure(figsize=(7 * n_cols, 6 * n_rows))

# GridSpec: baseline | cbar_gap | 4 diff cols | cbar_gap
# Wider gap after baseline cbar so it doesn't overlap col 2
gs = fig.add_gridspec(
    n_rows, 8,
    width_ratios=[1, 0.04, 0.06, 1, 1, 1, 1, 0.04],
    wspace=0.02, hspace=0.18,
    left=0.09, right=0.97, top=0.93, bottom=0.04)

# Column mapping: data col index -> gridspec col index
# col 0 (baseline) -> gs 0
# col 1 (ssp585)   -> gs 3
# col 2 (os)       -> gs 4
# col 3 (sai 8.5)  -> gs 5
# col 4 (sai os)   -> gs 6
col_to_gs = {0: 0, 1: 3, 2: 4, 3: 5, 4: 6}

for ri, metric in enumerate(fig2_metrics):
    cfg = metrics_config.get(metric, {'label': metric, 'units': '', 'cmap_type': 'temp_increase'})
    diff_cmap     = get_diff_cmap(cfg['cmap_type'])
    baseline_cmap = get_baseline_cmap(cfg['cmap_type'])

    base_field = np.squeeze(baseline_data_fig2.get(metric))
    if base_field is None:
        continue

    abs_vals = base_field[land_mask]
    vmin_abs = np.nanpercentile(abs_vals, 2)
    vmax_abs = np.nanpercentile(abs_vals, 98)

    diff_fields = []
    for scen_key, _, is_base in columns:
        if is_base or scen_key is None:
            continue
        fut = overshoot_data_fig2.get(scen_key, {}).get(metric)
        if fut is not None:
            diff_fields.append((np.squeeze(fut) - base_field)[land_mask])
    vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_fields)), 95) if diff_fields else 1

    im_abs = im_diff = None

    for ci, (scen_key, col_title, is_base) in enumerate(columns):
        gs_col = col_to_gs[ci]
        ax = fig.add_subplot(gs[ri, gs_col], projection=ccrs.PlateCarree())
        setup_africa_map(ax)

        if is_base:
            pdata = base_field.copy()
            pdata[~land_mask] = np.nan
            im_abs = ax.pcolormesh(lon, lat, pdata, transform=ccrs.PlateCarree(),
                                   cmap=baseline_cmap, vmin=vmin_abs, vmax=vmax_abs)
        else:
            fut = overshoot_data_fig2.get(scen_key, {}).get(metric)
            if fut is not None:
                diff = np.squeeze(fut) - base_field
                diff[~land_mask] = np.nan
                im_diff = ax.pcolormesh(lon, lat, diff, transform=ccrs.PlateCarree(),
                                        cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff)

                # Hatch non-significant areas
                if (scen_key in ensemble_overshoot and
                    'ssp585' in ensemble_baseline_15C and
                    metric in ensemble_baseline_15C.get('ssp585', {}) and
                    metric in ensemble_overshoot.get(scen_key, {})):
                    sig_mask, _ = compute_significance_map(
                        ensemble_baseline_15C['ssp585'][metric],
                        ensemble_overshoot[scen_key][metric])
                    if sig_mask is not None:
                        add_hatching_nonsig(ax, lon, lat, sig_mask, land_mask)

        if ci == 0:
            ax.text(-0.28, 0.5, cfg['label'], transform=ax.transAxes,
                    fontsize=26, fontweight='bold', rotation=90,
                    va='center', ha='center')
        if ri == 0:
            ax.set_title(col_title, fontsize=22, fontweight='bold', pad=12)

    # Baseline colorbar (gs col 1)
    if im_abs is not None:
        cax = fig.add_subplot(gs[ri, 1])
        cb = plt.colorbar(im_abs, cax=cax, extend='both', extendfrac=0.05)
        cb.set_label(cfg['units'], fontsize=22, fontweight='bold')
        cb.ax.tick_params(labelsize=18)

    # Diff colorbar (gs col 7)
    if im_diff is not None:
        cax = fig.add_subplot(gs[ri, 7])
        cb = plt.colorbar(im_diff, cax=cax, extend='both', extendfrac=0.05)
        cb.set_label(f'\u0394 {cfg["units"]}', fontsize=22, fontweight='bold')
        cb.ax.tick_params(labelsize=18)

# Hatching legend at bottom
fig.legend(
    handles=[Patch(facecolor='white', edgecolor='gray',
                   hatch='////', label='Not significant (p \u2265 0.05)')],
    loc='lower center', ncol=1, fontsize=22, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.01))

fig.suptitle(
    'Climate Index Changes: Overshoot (2060\u20132079) Relative to Baseline (2015\u20132034)',
    fontsize=30, fontweight='bold')

fname = f'{output_dir}/fig2_spatial_maps_4indices_5cols.png'
plt.savefig(fname, dpi=700, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname}')
print('Fig 2 complete.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Scenario pairs: (baseline_key, future_key, display_label) ─────────
# All use 1.5°C baseline (ssp585, 2015-2034) vs overshoot (2060-2079)
scenario_pairs_15C = [
    ('ssp585', 'ssp585',                    'SSP5-8.5'),
    ('ssp585', 'full_ssp534OS',             'SSP5-3.4-OS'),
    ('ssp585', 'full_ssp585_15_feedback',   'SSP5-8.5-1.5 SAI'),
    ('ssp585', 'full_ssp534OS_15_feedback',  'SSP5-3.4-OS-1.5 SAI'),
]

# ── Ordered metrics (temperature then precipitation) ─────────────────
temp_metrics  = ['mean_temperature', 'annual_gdd', 'gdd_intensity',
                 'optimal_growing_days', 'wsdi']
precip_metrics = ['r1mm', 'rx5day', 'cwd', 'cdd', 'sdii', 'prcptot']
all_metrics = temp_metrics + precip_metrics

# ── Compute significance percentages ─────────────────────────────────
rows = []
for metric in all_metrics:
    cfg = metrics_config[metric]
    for base_scen, fut_scen, label in scenario_pairs_15C:
        sig_pct = np.nan
        if (base_scen in ensemble_baseline_15C and
            fut_scen in ensemble_overshoot and
            metric in ensemble_baseline_15C.get(base_scen, {}) and
            metric in ensemble_overshoot.get(fut_scen, {})):

            sig_mask, _ = compute_significance_map(
                ensemble_baseline_15C[base_scen][metric],
                ensemble_overshoot[fut_scen][metric]
            )
            if sig_mask is not None:
                sig_pct = np.sum(sig_mask & land_mask) / np.sum(land_mask) * 100

        rows.append({
            'metric': metric,
            'metric_label': cfg['label'],
            'scenario': label,
            'sig_pct': sig_pct
        })

df = pd.DataFrame(rows)

# ── Build the heatmap matrix ─────────────────────────────────────────
ordered_labels = [metrics_config[m]['label'] for m in all_metrics]
scenario_labels = [s[2] for s in scenario_pairs_15C]

heatmap_data = df.pivot(index='metric_label', columns='scenario', values='sig_pct')
heatmap_data = heatmap_data.reindex(index=ordered_labels, columns=scenario_labels)

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)

# Text annotations
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.iloc[i, j]
        if not np.isnan(val):
            text_color = 'white' if val > 65 else 'black'
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                    fontsize=10, fontweight='bold', color=text_color)

# Axes
ax.set_xticks(np.arange(len(scenario_labels)))
ax.set_yticks(np.arange(len(ordered_labels)))
ax.set_xticklabels(scenario_labels, fontsize=11, ha='center')
ax.set_yticklabels(ordered_labels, fontsize=10)
ax.set_xlabel('Scenario', fontsize=15, fontweight='bold')
ax.set_ylabel('Climate Index', fontsize=15, fontweight='bold')

# Horizontal separator between temperature and precipitation indices
n_temp = len(temp_metrics)
ax.axhline(y=n_temp - 0.5, color='white', linewidth=2.5)

# Vertical dashed separator between non-SAI and SAI columns
ax.axvline(x=1.5, color='white', linewidth=2.5, linestyle='--')

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('Significant Area (%)', fontsize=12)

ax.set_title(
    'Detectability of climatic changes over land compared to a 1.5C threshold',
    fontsize=14, fontweight='bold', pad=12
)

plt.tight_layout()
plt.savefig(f'{output_dir}/stats_significance_heatmap_15C_4scenarios.png',
            dpi=400, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: stats_significance_heatmap_15C_4scenarios.png')


## Surface Moisture Budget Analysis

This section computes and visualizes the surface moisture budget (P - ET) for African monsoonal regions:
- **West Africa (WAF)**: 4°N-18°N, 18°W-15°E
- **Central Africa (CAF)**: 10°S-8°N, 8°E-27°E  
- **East Africa (EAF)**: 12°S-12°N, 27°E-52°E

ET is derived from latent heat flux: ET (mm/day) = LHFLX (W/m²) / 2.5×10⁶ × 86400


In [ ]:
def compute_ET_from_LHFLX(lhflx):
    """Convert Latent Heat Flux (W/m²) to Evapotranspiration (mm/day)."""
    L_v = 2.5e6  # J/kg
    seconds_per_day = 86400
    ET = (lhflx / L_v) * seconds_per_day
    return ET

def parse_flux_filename(filepath):
    """Parse flux filename to extract scenario, variable, and ensemble info."""
    import re
    fname = os.path.basename(filepath)
    
    # Determine variable
    if 'LHFLX' in fname:
        var = 'LHFLX'
    elif 'SHFLX' in fname:
        var = 'SHFLX'
    else:
        return None
    
    # Determine scenario
    if 'historical' in fname.lower() or 'HIST' in fname:
        scenario = 'historical'
    elif 'feedback.15C' in fname and 'SSP5-8.5' in fname:
        scenario = 'ssp585_15_feedback'
    elif 'feedback.15C' in fname and 'SSP5-3.4OS' in fname:
        scenario = 'ssp534os_15_feedback'
    elif 'feedback.20C' in fname:
        scenario = 'ssp534os_20_feedback'
    elif 'SSP5-8.5' in fname:
        scenario = 'ssp585'
    elif 'SSP5-3.4OS' in fname:
        scenario = 'ssp534os'
    else:
        return None
    
    # Extract ensemble number
    ens_match = re.search(r'\.(\d{3})\.cam', fname)
    ensemble = ens_match.group(1) if ens_match else '001'
    
    return {'path': filepath, 'scenario': scenario, 'variable': var, 'ensemble': ensemble}

def create_regional_land_mask(lat, lon, region_bounds, global_land_mask):
    """Create a land mask for a specific region."""
    lat_min, lat_max = region_bounds['lat']
    lon_min, lon_max = region_bounds['lon']
    
    # Find indices
    lat_idx = np.where((lat >= lat_min) & (lat <= lat_max))[0]
    lon_idx = np.where((lon >= lon_min) & (lon <= lon_max))[0]
    
    # Extract regional land mask
    regional_mask = global_land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
    
    return regional_mask, slice(lat_idx[0], lat_idx[-1]+1), slice(lon_idx[0], lon_idx[-1]+1)

def compute_regional_weighted_mean(data, lat, land_mask):
    """Compute area-weighted mean over land points."""
    weights = np.cos(np.deg2rad(lat))[:, np.newaxis] * np.ones((1, data.shape[1]))
    weights = np.broadcast_to(weights, data.shape)
    masked_data = np.where(land_mask, data, np.nan)
    masked_weights = np.where(land_mask, weights, 0)
    
    weighted_sum = np.nansum(masked_data * masked_weights)
    weight_sum = np.nansum(masked_weights[~np.isnan(masked_data)])
    
    return weighted_sum / weight_sum if weight_sum > 0 else np.nan


In [ ]:
# Find all flux files
flux_files = glob.glob(os.path.join(flux_data_dir, '*.nc'))
print(f"Found {len(flux_files)} NetCDF files")

# Parse all files
flux_file_info = []
for f in flux_files:
    info = parse_flux_filename(f)
    if info:
        flux_file_info.append(info)

# Create summary dataframe
flux_df = pd.DataFrame(flux_file_info)
print("\nFile summary:")
print(flux_df.groupby(['scenario', 'variable']).size().unstack(fill_value=0))


In [ ]:
def load_flux_data_by_scenario(flux_df, scenario, variable='LHFLX'):
    """Load flux data for a scenario, keeping xarray structure with time coordinates."""
    subset = flux_df[(flux_df['scenario'] == scenario) & 
                     (flux_df['variable'] == variable)]
    
    if len(subset) == 0:
        print(f"  No files found for {scenario} - {variable}")
        return []
    
    all_data = []
    
    for _, row in subset.iterrows():
        try:
            ds = xr.open_dataset(row['path'])
            
            if variable in ds.data_vars:
                data = ds[variable]
                
                # Subset to Africa
                data = data.sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                )
                
                all_data.append(data.load())
                print(f"    Loaded ensemble {row['ensemble']}: {data.time.dt.year.values[0]}-{data.time.dt.year.values[-1]}")
            
            ds.close()
        except Exception as e:
            print(f"  Error loading {row['path']}: {e}")
    
    return all_data

def compute_et_period_means(flux_data_list, baseline_period, overshoot_period):
    """Compute period means for ET following the established analysis pattern."""
    if not flux_data_list:
        return None
    
    baseline_means = []
    overshoot_means = []
    
    for data in flux_data_list:
        # Convert LHFLX to ET
        et = compute_ET_from_LHFLX(data)
        
        # Get years
        years = et.time.dt.year.values
        
        # Baseline period
        baseline_mask = (years >= baseline_period[0]) & (years <= baseline_period[1])
        if np.sum(baseline_mask) > 0:
            baseline_means.append(np.nanmean(et.values[baseline_mask], axis=0))
        
        # Overshoot period
        overshoot_mask = (years >= overshoot_period[0]) & (years <= overshoot_period[1])
        if np.sum(overshoot_mask) > 0:
            overshoot_means.append(np.nanmean(et.values[overshoot_mask], axis=0))
    
    result = {}
    if baseline_means:
        result['baseline'] = np.nanmean(baseline_means, axis=0)
    if overshoot_means:
        result['overshoot'] = np.nanmean(overshoot_means, axis=0)
    
    return result if result else None

# Load LHFLX data for all scenarios
print("Loading LHFLX data...")

flux_scenarios = ['ssp585', 'ssp534os', 'ssp585_15_feedback', 
                  'ssp534os_15_feedback', 'ssp534os_20_feedback']

flux_raw_data = {}  # Store raw xarray data
flux_lat, flux_lon = None, None

for scenario in flux_scenarios:
    print(f"\nProcessing {scenario}...")
    data_list = load_flux_data_by_scenario(flux_df, scenario, 'LHFLX')
    
    if data_list:
        flux_raw_data[scenario] = data_list
        if flux_lat is None:
            flux_lat = data_list[0].lat.values
            flux_lon = data_list[0].lon.values

print(f"\nLoaded flux data for {len(flux_raw_data)} scenarios")

# prec_files is already loaded earlier in the notebook with scenario info
print(f"Using {len(prec_files)} precipitation files from prec_data_dir")

# Compute period means using the established periods
print("\nComputing ET period means...")
print(f"Baseline period: {analysis_periods['early_warming_15C']}")
print(f"Overshoot period: {analysis_periods['overshoot_period']}")

et_period_means = {}
for scenario, data_list in flux_raw_data.items():
    # Use early_warming_15C as baseline for most scenarios
    # SAI scenarios start in 2040, so use overshoot period for both baseline comparison
    if 'feedback' in scenario:
        baseline_period = analysis_periods['overshoot_period']  # Compare within overshoot
    else:
        baseline_period = analysis_periods['early_warming_15C']
    
    result = compute_et_period_means(data_list, baseline_period, analysis_periods['overshoot_period'])
    if result:
        et_period_means[scenario] = result
        print(f"  {scenario}: baseline shape={result.get('baseline', np.array([])).shape}, overshoot shape={result.get('overshoot', np.array([])).shape}")

print(f"\nComputed period means for {len(et_period_means)} scenarios")


In [ ]:
def compute_moisture_budget_period_means(flux_raw_data, flux_df, prec_files, 
                                          baseline_period, overshoot_period):
    """Compute P-ET (surface moisture budget) period means."""
    # Map flux scenario names to precipitation scenario names
    scenario_mapping = {
        'ssp585': 'ssp585',
        'ssp534os': 'ssp534OS',
        'ssp585_15_feedback': 'ssp585_15_feedback',
        'ssp534os_15_feedback': 'ssp534OS_15_feedback',
        'ssp534os_20_feedback': 'ssp534OS_20_feedback'
    }
    
    # Scenarios that start at 2040 and need ssp585 for baseline
    needs_ssp585_baseline = ['ssp534os', 'ssp534os_15_feedback', 'ssp534os_20_feedback']
    
    # Get ssp585 files for baseline merging
    ssp585_prec_files = [f for f in prec_files if f['scenario'] == 'ssp585']
    ssp585_flux_data = flux_raw_data.get('ssp585', [])
    
    moisture_budget = {}
    
    for flux_scenario, et_data_list in flux_raw_data.items():
        print(f"  Processing {flux_scenario}...")
        
        prec_scenario = scenario_mapping.get(flux_scenario, flux_scenario)
        
        # Find matching precipitation files for overshoot period
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        
        # For scenarios starting at 2040, use ssp585 for baseline
        use_ssp585_for_baseline = flux_scenario in needs_ssp585_baseline
        
        if not prec_scenario_files and not use_ssp585_for_baseline:
            print(f"    No precipitation files found for {prec_scenario}")
            continue
        
        print(f"    Found {len(prec_scenario_files)} precipitation files for {prec_scenario}")
        if use_ssp585_for_baseline:
            print(f"    Using ssp585 data for baseline (scenario starts at 2040)")
        
        baseline_vals = []
        overshoot_vals = []
        
        # Process each ensemble member
        n_ens = len(et_data_list)
        for ens_idx in range(n_ens):
            try:
                # For baseline: use ssp585 data if scenario starts at 2040
                if use_ssp585_for_baseline and ens_idx < len(ssp585_flux_data) and ens_idx < len(ssp585_prec_files):
                    # Use ssp585 for baseline
                    et_baseline_data = compute_ET_from_LHFLX(ssp585_flux_data[ens_idx])
                    prec_path_baseline = ssp585_prec_files[ens_idx]['path']
                    prec_ds_baseline = xr.open_dataset(prec_path_baseline)
                    prec_baseline_raw = prec_ds_baseline['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    )
                    prec_baseline_mm = prec_baseline_raw * 86400 * 1000
                    
                    et_years = et_baseline_data.time.dt.year.values
                    prec_years = prec_baseline_mm.time.dt.year.values
                    
                    baseline_mask_et = (et_years >= baseline_period[0]) & (et_years <= baseline_period[1])
                    baseline_mask_prec = (prec_years >= baseline_period[0]) & (prec_years <= baseline_period[1])
                    
                    if np.sum(baseline_mask_et) > 0 and np.sum(baseline_mask_prec) > 0:
                        et_baseline = np.nanmean(et_baseline_data.values[baseline_mask_et], axis=0)
                        prec_baseline = np.nanmean(prec_baseline_mm.values[baseline_mask_prec], axis=0)
                        p_et_baseline = prec_baseline - et_baseline
                        baseline_vals.append(p_et_baseline)
                    
                    prec_ds_baseline.close()
                    
                elif not use_ssp585_for_baseline and ens_idx < len(et_data_list) and ens_idx < len(prec_scenario_files):
                    # Use the scenario's own data for baseline
                    et_data = et_data_list[ens_idx]
                    et = compute_ET_from_LHFLX(et_data)
                    et_years = et.time.dt.year.values
                    
                    prec_path = prec_scenario_files[ens_idx]['path']
                    prec_ds = xr.open_dataset(prec_path)
                    prec = prec_ds['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    )
                    prec_mm = prec * 86400 * 1000
                    prec_years = prec_mm.time.dt.year.values
                    
                    baseline_mask_et = (et_years >= baseline_period[0]) & (et_years <= baseline_period[1])
                    baseline_mask_prec = (prec_years >= baseline_period[0]) & (prec_years <= baseline_period[1])
                    
                    if np.sum(baseline_mask_et) > 0 and np.sum(baseline_mask_prec) > 0:
                        et_baseline = np.nanmean(et.values[baseline_mask_et], axis=0)
                        prec_baseline = np.nanmean(prec_mm.values[baseline_mask_prec], axis=0)
                        p_et_baseline = prec_baseline - et_baseline
                        baseline_vals.append(p_et_baseline)
                    
                    prec_ds.close()
                
                # For overshoot: always use the scenario's own data
                if ens_idx < len(et_data_list) and ens_idx < len(prec_scenario_files):
                    et_data = et_data_list[ens_idx]
                    et = compute_ET_from_LHFLX(et_data)
                    et_years = et.time.dt.year.values
                    
                    prec_path = prec_scenario_files[ens_idx]['path']
                    prec_ds = xr.open_dataset(prec_path)
                    prec = prec_ds['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    )
                    prec_mm = prec * 86400 * 1000
                    prec_years = prec_mm.time.dt.year.values
                    
                    overshoot_mask_et = (et_years >= overshoot_period[0]) & (et_years <= overshoot_period[1])
                    overshoot_mask_prec = (prec_years >= overshoot_period[0]) & (prec_years <= overshoot_period[1])
                    
                    if np.sum(overshoot_mask_et) > 0 and np.sum(overshoot_mask_prec) > 0:
                        et_overshoot = np.nanmean(et.values[overshoot_mask_et], axis=0)
                        prec_overshoot = np.nanmean(prec_mm.values[overshoot_mask_prec], axis=0)
                        p_et_overshoot = prec_overshoot - et_overshoot
                        overshoot_vals.append(p_et_overshoot)
                    
                    prec_ds.close()
                    
            except Exception as e:
                print(f"    Error processing ensemble {ens_idx}: {e}")
                continue
        
        # Compute ensemble means AFTER computing P-ET for each member
        if baseline_vals:
            if flux_scenario not in moisture_budget:
                moisture_budget[flux_scenario] = {}
            moisture_budget[flux_scenario]['baseline'] = np.nanmean(baseline_vals, axis=0)
            print(f"    Baseline: {len(baseline_vals)} ensemble members")
        
        if overshoot_vals:
            if flux_scenario not in moisture_budget:
                moisture_budget[flux_scenario] = {}
            moisture_budget[flux_scenario]['overshoot'] = np.nanmean(overshoot_vals, axis=0)
            print(f"    Overshoot: {len(overshoot_vals)} ensemble members")
    
    return moisture_budget

def plot_moisture_budget_spatial(moisture_budget_15C, moisture_budget_20C, flux_lat, flux_lon, land_mask, output_dir):
    """Plot surface moisture budget for monsoonal regions."""
    print("\nCreating spatial moisture budget plots...")
    
    def plot_single_threshold(moisture_budget, threshold_label, baseline_period, filename):
        """Plot a single threshold (1.5°C or 2.0C)."""
        if 'ssp585' not in moisture_budget or 'baseline' not in moisture_budget['ssp585']:
            print(f"  No {threshold_label} baseline data available")
            return
        
        n_regions = len(MONSOONAL_REGIONS)
        
        # Layout: baseline | cbar | diff1 | diff2 | cbar
        fig = plt.figure(figsize=(28, 5 * n_regions))
        gs = fig.add_gridspec(n_regions, 5, width_ratios=[0.7, 0.015, 0.7, 0.7, 0.015], 
                              wspace=0.10, hspace=0.12,
                              left=0.08, right=0.95, top=0.90, bottom=0.04)
        
        for ri, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
            lat_min, lat_max = region_bounds['lat']
            lon_min, lon_max = region_bounds['lon']
            abbrev = region_bounds['abbrev']
            
            lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
            lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
            
            if len(lat_idx) == 0 or len(lon_idx) == 0:
                continue
            
            reg_lat = flux_lat[lat_idx]
            reg_lon = flux_lon[lon_idx]
            reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
            
            baseline = moisture_budget['ssp585']['baseline'][lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
            ssp585_ov = moisture_budget['ssp585'].get('overshoot')
            ssp534os_ov = moisture_budget.get('ssp534os', {}).get('overshoot')
            
            if ssp585_ov is not None: ssp585_ov = ssp585_ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
            if ssp534os_ov is not None: ssp534os_ov = ssp534os_ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
            
            # Color limits
            vmin_abs = np.nanpercentile(baseline[reg_land_mask], 2)
            vmax_abs = np.nanpercentile(baseline[reg_land_mask], 98)
            
            diff_vals = []
            if ssp585_ov is not None: diff_vals.append((ssp585_ov - baseline)[reg_land_mask])
            if ssp534os_ov is not None: diff_vals.append((ssp534os_ov - baseline)[reg_land_mask])
            vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_vals)), 95) if diff_vals else 1
            
            im_abs, im_diff = None, None
            
            # Column 0: Baseline
            ax = fig.add_subplot(gs[ri, 0], projection=ccrs.PlateCarree())
            ax.set_facecolor('lightgray')
            pdata = np.where(reg_land_mask, baseline, np.nan)
            im_abs = ax.pcolormesh(reg_lon, reg_lat, pdata, transform=ccrs.PlateCarree(), 
                                   cmap='BrBG', vmin=vmin_abs, vmax=vmax_abs)
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.set_extent([lon_min, lon_max, lat_min, lat_max])
            if ri == 0: ax.set_title(f'{threshold_label} Baseline\n({baseline_period[0]}-{baseline_period[1]})', fontsize=10, fontweight='bold')
            ax.text(-0.12, 0.5, f'{region_name}\n({abbrev})', transform=ax.transAxes, fontsize=13, 
                    fontweight='bold', rotation=90, va='center', ha='center')
            
            # Column 1: Baseline colorbar
            cax = fig.add_subplot(gs[ri, 1])
            cb = plt.colorbar(im_abs, cax=cax, extend='both')
            cb.ax.tick_params(labelsize=11)
            
            # Column 2: SSP5-8.5 diff
            ax = fig.add_subplot(gs[ri, 2], projection=ccrs.PlateCarree())
            ax.set_facecolor('lightgray')
            if ssp585_ov is not None:
                diff = np.where(reg_land_mask, ssp585_ov - baseline, np.nan)
                im_diff = ax.pcolormesh(reg_lon, reg_lat, diff, transform=ccrs.PlateCarree(), 
                                        cmap='BrBG', vmin=-vmax_diff, vmax=vmax_diff)
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.set_extent([lon_min, lon_max, lat_min, lat_max])
            if ri == 0: ax.set_title(f'{threshold_label} SSP5-8.5\n- Baseline', fontsize=10, fontweight='bold')
            
            # Column 3: SSP5-3.4-OS diff
            ax = fig.add_subplot(gs[ri, 3], projection=ccrs.PlateCarree())
            ax.set_facecolor('lightgray')
            if ssp534os_ov is not None:
                diff = np.where(reg_land_mask, ssp534os_ov - baseline, np.nan)
                im_diff = ax.pcolormesh(reg_lon, reg_lat, diff, transform=ccrs.PlateCarree(), 
                                        cmap='BrBG', vmin=-vmax_diff, vmax=vmax_diff)
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.set_extent([lon_min, lon_max, lat_min, lat_max])
            if ri == 0: ax.set_title(f'{threshold_label} SSP5-3.4-OS\n- Baseline', fontsize=10, fontweight='bold')
            
            # Column 4: Diff colorbar
            if im_diff is not None:
                cax = fig.add_subplot(gs[ri, 4])
                cb = plt.colorbar(im_diff, cax=cax, extend='both')
                cb.set_label('Δ mm/day', fontsize=13, fontweight='bold')
                cb.ax.tick_params(labelsize=11)
        
        fig.suptitle(f'Surface Moisture Budget - {threshold_label} Threshold\n(Overshoot {analysis_periods["overshoot_period"][0]}-{analysis_periods["overshoot_period"][1]} - Baseline)', 
                     fontsize=13, fontweight='bold')
        
        plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"  Saved: {filename}")
    
    # Plot 1.5°C threshold
    plot_single_threshold(moisture_budget_15C, '1.5°C', analysis_periods['early_warming_15C'], 'moisture_budget_spatial_15C.png')
    
    # Plot 2.0°C threshold
    plot_single_threshold(moisture_budget_20C, '2.0°C', analysis_periods['early_warming_20C'], 'moisture_budget_spatial_20C.png')

def plot_moisture_budget_sai_impact(moisture_budget_15C, moisture_budget_20C, flux_lat, flux_lon, land_mask, output_dir):
    """Plot SAI impact on moisture budget following established SAI impact layout."""
    print("\nCreating SAI impact moisture budget plot...")
    
    comparisons = [
        ('ssp585_15_feedback', 'ssp585', 'SAI 1.5°C\n- SSP5-8.5', '15C'),
        ('ssp534os_15_feedback', 'ssp534os', 'SAI 1.5°C\n- SSP5-3.4-OS', '15C'),
        ('ssp534os_20_feedback', 'ssp585', 'SAI 2.0°C\n- SSP5-8.5', '20C'),
        ('ssp534os_20_feedback', 'ssp534os', 'SAI 2.0°C\n- SSP5-3.4-OS', '20C')
    ]
    
    n_regions = len(MONSOONAL_REGIONS)
    
    fig = plt.figure(figsize=(28, 5 * n_regions))
    gs = fig.add_gridspec(n_regions, 5, width_ratios=[1, 1, 1, 1, 0.03], 
                          wspace=0.02, hspace=0.12,
                          left=0.08, right=0.95, top=0.90, bottom=0.04)
    
    for ri, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
        lat_min, lat_max = region_bounds['lat']
        lon_min, lon_max = region_bounds['lon']
        abbrev = region_bounds['abbrev']
        
        lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
        lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
        
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            continue
        
        reg_lat = flux_lat[lat_idx]
        reg_lon = flux_lon[lon_idx]
        reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
        
        # Calculate vmax from all comparisons
        all_diffs = []
        for sai_scen, ref_scen, _, thresh in comparisons:
            mb = moisture_budget_15C if thresh == '15C' else moisture_budget_20C
            if sai_scen in mb and 'overshoot' in mb[sai_scen]:
                if ref_scen in mb and 'overshoot' in mb[ref_scen]:
                    sai_data = mb[sai_scen]['overshoot'][lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    ref_data = mb[ref_scen]['overshoot'][lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    diff = sai_data - ref_data
                    all_diffs.append(diff[reg_land_mask])
        
        vmax = np.nanpercentile(np.abs(np.concatenate(all_diffs)), 95) if all_diffs else 1
        im = None
        
        for ci, (sai_scen, ref_scen, title, thresh) in enumerate(comparisons):
            ax = fig.add_subplot(gs[ri, ci], projection=ccrs.PlateCarree())
            ax.set_facecolor('lightgray')
            
            mb = moisture_budget_15C if thresh == '15C' else moisture_budget_20C
            if sai_scen in mb and 'overshoot' in mb[sai_scen]:
                if ref_scen in mb and 'overshoot' in mb[ref_scen]:
                    sai_data = mb[sai_scen]['overshoot'][lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    ref_data = mb[ref_scen]['overshoot'][lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    diff = np.where(reg_land_mask, sai_data - ref_data, np.nan)
                    im = ax.pcolormesh(reg_lon, reg_lat, diff, transform=ccrs.PlateCarree(), 
                                       cmap='BrBG', vmin=-vmax, vmax=vmax)
            
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.set_extent([lon_min, lon_max, lat_min, lat_max])
            
            if ri == 0:
                ax.set_title(title, fontsize=10, fontweight='bold')
            if ci == 0:
                ax.text(-0.15, 0.5, f'{region_name}\n({abbrev})', transform=ax.transAxes, fontsize=14, 
                        fontweight='bold', rotation=90, va='center', ha='center')
        
        if im is not None:
            cax = fig.add_subplot(gs[ri, 4])
            cb = plt.colorbar(im, cax=cax, extend='both', extendfrac=0.05)
            cb.set_label('Δ mm/day', fontsize=13, fontweight='bold')
            cb.ax.tick_params(labelsize=11)
    
    fig.suptitle(f'SAI Impact on Surface Moisture Budget\n(Overshoot Period {analysis_periods["overshoot_period"][0]}-{analysis_periods["overshoot_period"][1]})', 
                 fontsize=14, fontweight='bold')
    filename = 'moisture_budget_sai_impact.png'
    plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  Saved: {filename}")

# Compute moisture budget for BOTH threshold baselines
print("\nComputing surface moisture budget...")

# 1.5°C baseline
print(f"\n1.5°C Baseline period: {analysis_periods['early_warming_15C']}")
print(f"Overshoot period: {analysis_periods['overshoot_period']}")
moisture_budget_15C = compute_moisture_budget_period_means(
    flux_raw_data, flux_df, prec_files,
    analysis_periods['early_warming_15C'], 
    analysis_periods['overshoot_period']
)
print(f"Computed 1.5°C moisture budget for {len(moisture_budget_15C)} scenarios")

# 2.0°C baseline
print(f"\n2.0°C Baseline period: {analysis_periods['early_warming_20C']}")
moisture_budget_20C = compute_moisture_budget_period_means(
    flux_raw_data, flux_df, prec_files,
    analysis_periods['early_warming_20C'], 
    analysis_periods['overshoot_period']
)
print(f"Computed 2.0°C moisture budget for {len(moisture_budget_20C)} scenarios")

# Run spatial plots
if moisture_budget_15C and moisture_budget_20C and flux_lat is not None:
    plot_moisture_budget_spatial(moisture_budget_15C, moisture_budget_20C, flux_lat, flux_lon, land_mask, output_dir)
    plot_moisture_budget_sai_impact(moisture_budget_15C, moisture_budget_20C, flux_lat, flux_lon, land_mask, output_dir)
else:
    print("No moisture budget data available for spatial plots")


In [ ]:
# COMBINED: Time Series (row 1) + Seasonal Bar (row 2)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

n_regions = len(MONSOONAL_REGIONS)

fig, axes = plt.subplots(2, n_regions, figsize=(6 * n_regions, 16),
                         gridspec_kw={'hspace': 0.35})
fig.patch.set_facecolor('white')

# ── Scenario definitions ──
ts_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728', '-'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4', '-'),
    ('ssp585_15_feedback', 'ssp585_15_feedback', 'SSP5-8.5 1.5\u00b0C SAI', '#d62728', '--'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SSP5-3.4-OS 1.5\u00b0C SAI', '#1f77b4', '--'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SSP5-3.4-OS 2.0\u00b0C SAI', '#2ca02c', '--'),
]

bar_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SAI 1.5\u00b0C', '#2ca02c'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SAI 2.0\u00b0C', '#9467bd'),
]

seasons = {'DJF': [12, 1, 2], 'MAM': [3, 4, 5], 'JJA': [6, 7, 8], 'SON': [9, 10, 11]}
season_order = ['DJF', 'MAM', 'JJA', 'SON']
x_pos = np.arange(len(season_order))
bar_width = 0.2
overshoot_start, overshoot_end = analysis_periods['overshoot_period']

for idx, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
    lat_min, lat_max = region_bounds['lat']
    lon_min, lon_max = region_bounds['lon']

    lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
    lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
    if len(lat_idx) == 0 or len(lon_idx) == 0:
        continue

    reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
    reg_lat = flux_lat[lat_idx]
    weights = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
    weights = np.broadcast_to(weights, reg_land_mask.shape)

    # ROW 0: TIME SERIES
    ax_ts = axes[0, idx]
    ax_ts.set_facecolor('white')

    for flux_scenario, prec_scenario, label, color, ls in ts_scenarios:
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_years = None
        all_vals = []
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_ds.close()

                common_years = np.intersect1d(et_years, prec_years)
                if len(common_years) == 0:
                    continue
                p_et_ts, years_ts = [], []
                for year in common_years:
                    et_mask = et_years == year
                    prec_mask = prec_years == year
                    if np.sum(et_mask) > 0 and np.sum(prec_mask) > 0:
                        et_yr = np.nanmean(et_reg[et_mask], axis=0)
                        pr_yr = np.nanmean(prec_reg[prec_mask], axis=0)
                        p_et = pr_yr - et_yr
                        masked = np.where(reg_land_mask, p_et, np.nan)
                        w = np.where(reg_land_mask, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            p_et_ts.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                            years_ts.append(year)

                if p_et_ts:
                    years_arr = np.array(years_ts)
                    vals_arr = np.array(p_et_ts)
                    keep = years_arr <= 2099
                    if np.sum(keep) > 0:
                        if all_years is None:
                            all_years = years_arr[keep]
                        all_vals.append(vals_arr[keep])
            except:
                continue

        if all_vals:
            minlen = min(len(v) for v in all_vals)
            arr = np.array([v[:minlen] for v in all_vals])
            mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
            ax_ts.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
            if len(arr) > 1:
                std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax_ts.fill_between(all_years[:minlen], mean - std, mean + std, color=color, alpha=0.15)

    ax_ts.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax_ts.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=14, fontweight='bold')
    ax_ts.set_xlim(2015, 2100)
    ax_ts.set_ylabel('mm/day', fontsize=22, fontweight='bold')
    ax_ts.tick_params(labelsize=11)
    ax_ts.grid(True, alpha=0.3, linestyle='--')
    if idx == 0:
        ax_ts.legend(fontsize=7, loc='upper left', framealpha=0.3)

    # ROW 1: SEASONAL BAR
    ax_bar = axes[1, idx]
    ax_bar.set_facecolor('white')

    for si, (flux_scenario, prec_scenario, label, color) in enumerate(bar_scenarios):
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_seasonal_means = {s: [] for s in season_order}
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                et_months = et.time.dt.month.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_months = prec_mm.time.dt.month.values
                prec_ds.close()

                et_ov = (et_years >= overshoot_start) & (et_years <= overshoot_end)
                prec_ov = (prec_years >= overshoot_start) & (prec_years <= overshoot_end)

                for season in season_order:
                    sm = seasons[season]
                    et_sm = et_ov & np.isin(et_months, sm)
                    prec_sm = prec_ov & np.isin(prec_months, sm)
                    if np.sum(et_sm) > 0 and np.sum(prec_sm) > 0:
                        et_s = np.nanmean(et_reg[et_sm], axis=0)
                        pr_s = np.nanmean(prec_reg[prec_sm], axis=0)
                        p_et = pr_s - et_s
                        masked = np.where(reg_land_mask, p_et, np.nan)
                        w = np.where(reg_land_mask, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            all_seasonal_means[season].append(
                                np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
            except:
                continue

        seasonal_vals = [np.nanmean(all_seasonal_means[s]) if all_seasonal_means[s] else np.nan
                         for s in season_order]
        ax_bar.bar(x_pos + si * bar_width, seasonal_vals, bar_width,
                   label=label, color=color, alpha=0.8)

    ax_bar.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax_bar.set_xlabel('Season', fontsize=22, fontweight='bold')
    ax_bar.set_ylabel('mm/day', fontsize=22, fontweight='bold')
    ax_bar.set_xticks(x_pos + 1.5 * bar_width)
    ax_bar.set_xticklabels(season_order, fontsize=11)
    ax_bar.tick_params(labelsize=11)
    ax_bar.grid(True, alpha=0.3, axis='y', linestyle='--')
    if idx == 0:
        ax_bar.legend(fontsize=8, framealpha=0.3)

# Row labels on left side
axes[0, 0].text(-0.18, 0.5, 'Annual Time Series\n(5-yr running mean)',
                transform=axes[0, 0].transAxes, fontsize=16, fontweight='bold',
                rotation=90, va='center', ha='center')
axes[1, 0].text(-0.18, 0.5, f'Seasonal\n({overshoot_start}\u2013{overshoot_end})',
                transform=axes[1, 0].transAxes, fontsize=16, fontweight='bold',
                rotation=90, va='center', ha='center')

fig.suptitle('Surface Moisture Budget: African Monsoonal Regions',
             fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0.03, 0, 1, 0.95])

fname = f'{output_dir}/moisture_budget_combined_ts_seasonal.png'
plt.savefig(fname, dpi=700, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_moisture_budget_regions(regions_dict, title_suffix, fname_suffix):
    """Plot moisture budget spatial maps for given regions."""

    col_defs = [
        (None,                      'Baseline\n(2015\u20132034)',  True),
        ('ssp585',                  'SSP5-8.5',               False),
        ('ssp534os',                'SSP5-3.4-OS',            False),
        ('ssp585_15_feedback',      'SSP5-8.5-1.5 SAI',      False),
        ('ssp534os_15_feedback',    'SSP5-3.4-OS-1.5 SAI',   False),
    ]

    regions = list(regions_dict.items())
    n_rows = len(regions)
    n_cols = len(col_defs)

    fig = plt.figure(figsize=(6 * n_cols + 2, 6 * n_rows))
    gs = fig.add_gridspec(
        n_rows, 8,
        width_ratios=[1, 0.035, 0.12, 1, 1, 1, 1, 0.035],
        wspace=0.05, hspace=0.18,
        left=0.06, right=0.97, top=0.92, bottom=0.04)

    col_to_gs = {0: 0, 1: 3, 2: 4, 3: 5, 4: 6}

    for ri, (region_name, region_bounds) in enumerate(regions):
        lat_min, lat_max = region_bounds['lat']
        lon_min, lon_max = region_bounds['lon']
        abbrev = region_bounds['abbrev']

        lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
        lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            continue

        reg_lat = flux_lat[lat_idx]
        reg_lon = flux_lon[lon_idx]
        reg_land = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]

        base_full = moisture_budget_15C.get('ssp585', {}).get('baseline')
        if base_full is None:
            continue
        base_reg = base_full[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]

        abs_vals = base_reg[reg_land]
        vmin_abs = np.nanpercentile(abs_vals, 2)
        vmax_abs = np.nanpercentile(abs_vals, 98)

        diff_all = []
        for scen_key, _, is_base in col_defs:
            if is_base or scen_key is None:
                continue
            ov = moisture_budget_15C.get(scen_key, {}).get('overshoot')
            if ov is not None:
                ov_reg = ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                diff_all.append((ov_reg - base_reg)[reg_land])
        vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_all)), 95) if diff_all else 1

        im_abs = im_diff = None

        for ci, (scen_key, col_title, is_base) in enumerate(col_defs):
            gs_col = col_to_gs[ci]
            ax = fig.add_subplot(gs[ri, gs_col], projection=ccrs.PlateCarree())
            ax.set_facecolor('lightgray')
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.set_extent([lon_min, lon_max, lat_min, lat_max])
            ax.tick_params(labelsize=12)

            if is_base:
                pdata = np.where(reg_land, base_reg, np.nan)
                im_abs = ax.pcolormesh(reg_lon, reg_lat, pdata,
                                       transform=ccrs.PlateCarree(),
                                       cmap='BrBG', vmin=vmin_abs, vmax=vmax_abs)
            else:
                ov = moisture_budget_15C.get(scen_key, {}).get('overshoot')
                if ov is not None:
                    ov_reg = ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    diff = np.where(reg_land, ov_reg - base_reg, np.nan)
                    im_diff = ax.pcolormesh(reg_lon, reg_lat, diff,
                                            transform=ccrs.PlateCarree(),
                                            cmap='BrBG', vmin=-vmax_diff, vmax=vmax_diff)

            if ci == 0:
                ax.text(-0.18, 0.5, f'{region_name}\n({abbrev})',
                        transform=ax.transAxes, fontsize=16, fontweight='bold',
                        rotation=90, va='center', ha='center')
            if ri == 0:
                ax.set_title(col_title, fontsize=15, fontweight='bold', pad=10)

        if im_abs is not None:
            cax = fig.add_subplot(gs[ri, 1])
            cb = plt.colorbar(im_abs, cax=cax, extend='both', extendfrac=0.05)
            cb.set_label('mm/day', fontsize=14, fontweight='bold')
            cb.ax.tick_params(labelsize=12)

        if im_diff is not None:
            cax = fig.add_subplot(gs[ri, 7])
            cb = plt.colorbar(im_diff, cax=cax, extend='both', extendfrac=0.05)
            cb.set_label('\u0394 mm/day', fontsize=14, fontweight='bold')
            cb.ax.tick_params(labelsize=12)

    fig.suptitle(
        f'Surface Moisture Budget {title_suffix}\n'
        f'Overshoot (2060\u20132079) Relative to Baseline (2015\u20132034)',
        fontsize=20, fontweight='bold', y=1.02)

    fname = f'{output_dir}/moisture_budget_spatial_{fname_suffix}.png'
    plt.savefig(fname, dpi=400, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f'Saved: {fname}')

print('Plotting function defined.')


In [ ]:
# SPATIAL MOISTURE BUDGET: 4 Scenarios over WAF, CAF, EAF

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

col_defs = [
    (None,                      'Baseline\n(2015\u20132034)',  True),
    ('ssp585',                  'SSP5-8.5',               False),
    ('ssp534os',                'SSP5-3.4-OS',            False),
    ('ssp585_15_feedback',      'SSP5-8.5-1.5 SAI',      False),
    ('ssp534os_15_feedback',    'SSP5-3.4-OS-1.5 SAI',   False),
]

regions = list(MONSOONAL_REGIONS.items())
n_rows = len(regions)
n_cols = len(col_defs)

fig = plt.figure(figsize=(6 * n_cols + 2, 6 * n_rows))

# GridSpec with generous spacing:
# col 0: baseline map
# col 1: baseline cbar (narrow)
# col 2: spacer (wider gap to prevent overlap)
# cols 3-6: four diff maps
# col 7: diff cbar (narrow)
gs = fig.add_gridspec(
    n_rows, 8,
    width_ratios=[1, 0.035, 0.12, 1, 1, 1, 1, 0.035],
    wspace=0.05, hspace=0.18,
    left=0.06, right=0.97, top=0.92, bottom=0.04)

col_to_gs = {0: 0, 1: 3, 2: 4, 3: 5, 4: 6}

for ri, (region_name, region_bounds) in enumerate(regions):
    lat_min, lat_max = region_bounds['lat']
    lon_min, lon_max = region_bounds['lon']
    abbrev = region_bounds['abbrev']

    lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
    lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
    if len(lat_idx) == 0 or len(lon_idx) == 0:
        continue

    reg_lat = flux_lat[lat_idx]
    reg_lon = flux_lon[lon_idx]
    reg_land = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]

    base_full = moisture_budget_15C.get('ssp585', {}).get('baseline')
    if base_full is None:
        continue
    base_reg = base_full[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]

    abs_vals = base_reg[reg_land]
    vmin_abs = np.nanpercentile(abs_vals, 2)
    vmax_abs = np.nanpercentile(abs_vals, 98)

    diff_all = []
    for scen_key, _, is_base in col_defs:
        if is_base or scen_key is None:
            continue
        ov = moisture_budget_15C.get(scen_key, {}).get('overshoot')
        if ov is not None:
            ov_reg = ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
            diff_all.append((ov_reg - base_reg)[reg_land])
    vmax_diff = np.nanpercentile(np.abs(np.concatenate(diff_all)), 95) if diff_all else 1

    im_abs = im_diff = None

    for ci, (scen_key, col_title, is_base) in enumerate(col_defs):
        gs_col = col_to_gs[ci]
        ax = fig.add_subplot(gs[ri, gs_col], projection=ccrs.PlateCarree())
        ax.set_facecolor('lightgray')
        ax.coastlines(linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max])

        # Increase tick label size on axes
        ax.tick_params(labelsize=12)

        if is_base:
            pdata = np.where(reg_land, base_reg, np.nan)
            im_abs = ax.pcolormesh(reg_lon, reg_lat, pdata,
                                   transform=ccrs.PlateCarree(),
                                   cmap='BrBG', vmin=vmin_abs, vmax=vmax_abs)
        else:
            ov = moisture_budget_15C.get(scen_key, {}).get('overshoot')
            if ov is not None:
                ov_reg = ov[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                diff = np.where(reg_land, ov_reg - base_reg, np.nan)
                im_diff = ax.pcolormesh(reg_lon, reg_lat, diff,
                                        transform=ccrs.PlateCarree(),
                                        cmap='BrBG', vmin=-vmax_diff, vmax=vmax_diff)

        # Row label
        if ci == 0:
            ax.text(-0.18, 0.5, f'{region_name}\n({abbrev})',
                    transform=ax.transAxes, fontsize=20, fontweight='bold',
                    rotation=90, va='center', ha='center')

        # Column title (larger, bolder)
        if ri == 0:
            ax.set_title(col_title, fontsize=19, fontweight='bold', pad=10)

    # Baseline colorbar (gs col 1)
    if im_abs is not None:
        cax = fig.add_subplot(gs[ri, 1])
        cb = plt.colorbar(im_abs, cax=cax, extend='both', extendfrac=0.05)
        cb.set_label('mm/day', fontsize=18, fontweight='bold')
        cb.ax.tick_params(labelsize=16)

    # Diff colorbar (gs col 7)
    if im_diff is not None:
        cax = fig.add_subplot(gs[ri, 7])
        cb = plt.colorbar(im_diff, cax=cax, extend='both', extendfrac=0.05)
        cb.set_label('\u0394 mm/day', fontsize=18, fontweight='bold')
        cb.ax.tick_params(labelsize=16)

fig.suptitle(
    'Surface Moisture Budget: Overshoot (2060\u20132079) Relative to Baseline (2015\u20132034)',
    fontsize=24, fontweight='bold', y=0.97)

fname = f'{output_dir}/moisture_budget_spatial_4scenarios.png'
plt.savefig(fname, dpi=400, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname}')


In [ ]:
def plot_moisture_budget_timeseries(flux_raw_data, prec_files, flux_lat, flux_lon, land_mask, output_dir):
    """Plot P-ET time series for each monsoonal region with ensemble spread."""
    scenarios_plot = [
        ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728', '-'),
        ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4', '-'),
        ('ssp585_15_feedback', 'ssp585_15_feedback', 'SSP5-8.5 1.5°C SAI', '#d62728', '--'),
        ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SSP5-3.4-OS 1.5°C SAI', '#1f77b4', '--'),
        ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SSP5-3.4-OS 2.0°C SAI', '#2ca02c', '--'),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    fig.patch.set_facecolor('white')
    
    for idx, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
        ax = axes[idx]
        ax.set_facecolor('white')
        
        lat_min, lat_max = region_bounds['lat']
        lon_min, lon_max = region_bounds['lon']
        
        # Find indices for this region
        lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
        lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
        
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            continue
        
        # Regional land mask and weights
        reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
        reg_lat = flux_lat[lat_idx]
        weights = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
        weights = np.broadcast_to(weights, reg_land_mask.shape)
        
        for flux_scenario, prec_scenario, label, color, ls in scenarios_plot:
            if flux_scenario not in flux_raw_data:
                continue
            
            data_list = flux_raw_data[flux_scenario]
            prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
            
            if not prec_scenario_files:
                continue
            
            # Collect annual P-ET means for each ensemble member
            all_years = None
            all_vals = []
            
            for ens_idx, et_data in enumerate(data_list):
                if ens_idx >= len(prec_scenario_files):
                    break
                
                try:
                    # Convert LHFLX to ET
                    et = compute_ET_from_LHFLX(et_data)
                    et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    et_years = et.time.dt.year.values
                    
                    # Load precipitation - use ['path'] since prec_scenario_files contains dicts
                    prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                    prec = prec_ds['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    )
                    prec_mm = prec * 86400 * 1000  # m/s to mm/day
                    prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    prec_years = prec_mm.time.dt.year.values
                    prec_ds.close()
                    
                    # Find common years
                    common_years = np.intersect1d(et_years, prec_years)
                    if len(common_years) == 0:
                        continue
                    
                    # Compute P-ET for each timestep
                    p_et_ts = []
                    years_ts = []
                    
                    for year in common_years:
                        et_mask = et_years == year
                        prec_mask = prec_years == year
                        
                        if np.sum(et_mask) > 0 and np.sum(prec_mask) > 0:
                            et_year_mean = np.nanmean(et_reg[et_mask], axis=0)
                            prec_year_mean = np.nanmean(prec_reg[prec_mask], axis=0)
                            p_et = prec_year_mean - et_year_mean
                            
                            # Compute weighted regional mean
                            masked = np.where(reg_land_mask, p_et, np.nan)
                            w = np.where(reg_land_mask, weights, 0)
                            if np.sum(~np.isnan(masked)) > 0:
                                p_et_ts.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                                years_ts.append(year)
                    
                    if p_et_ts:
                        # Filter to years <= 2099
                        years_arr = np.array(years_ts)
                        vals_arr = np.array(p_et_ts)
                        mask = years_arr <= 2099
                        
                        if np.sum(mask) > 0:
                            if all_years is None:
                                all_years = years_arr[mask]
                            all_vals.append(vals_arr[mask])
                
                except Exception as e:
                    continue
            
            if all_vals:
                # Handle different lengths - use minimum length
                minlen = min(len(v) for v in all_vals)
                arr = np.array([v[:minlen] for v in all_vals])
                
                # Compute ensemble mean and std with 5-year smoothing
                mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
                
                # Add ensemble spread if multiple members
                if len(arr) > 1:
                    std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                    ax.fill_between(all_years[:minlen], mean - std, mean + std, color=color, alpha=0.15)
        
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
        ax.set_xlabel('Year', fontsize=16, fontweight='bold')
        ax.set_ylabel('mm/day', fontsize=16, fontweight='bold')
        ax.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=12, fontweight='bold')
        ax.set_xlim(2015, 2100)
        
        # Set y-axis minimum per region
        ymin_limits = {'West Africa': 0.6, 'Central Africa': 1.0, 'East Africa': 0.25}
        if region_name in ymin_limits:
            ax.set_ylim(bottom=ymin_limits[region_name])
        
        ax.grid(True, alpha=0.3, linestyle='--')
        
        if idx == 0:
            ax.legend(fontsize=7, loc='upper left')
    
    fig.suptitle('Surface Moisture Budget Time Series - African Monsoonal Regions\n(5-year running mean, shading: ±1 std across ensembles)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    
    filename = 'moisture_budget_timeseries.png'
    plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Run time series plots
if flux_raw_data and flux_lat is not None:
    plot_moisture_budget_timeseries(flux_raw_data, prec_files, flux_lat, flux_lon, land_mask, output_dir)
else:
    print("No data available for time series plots")


In [ ]:
def plot_seasonal_moisture_budget(flux_raw_data, prec_files, flux_lat, flux_lon, land_mask, output_dir):
    """Plot seasonal moisture budget cycle for each monsoonal region."""
    seasons = {
        'DJF': [12, 1, 2],
        'MAM': [3, 4, 5],
        'JJA': [6, 7, 8],
        'SON': [9, 10, 11]
    }
    
    scenarios_plot = [
        ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728'),
        ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4'),
        ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SAI 1.5°C', '#2ca02c'),
        ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SAI 2.0°C', '#9467bd'),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    fig.patch.set_facecolor('white')
    
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    x_pos = np.arange(len(season_order))
    bar_width = 0.2
    
    overshoot_start, overshoot_end = analysis_periods['overshoot_period']
    
    for idx, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
        ax = axes[idx]
        ax.set_facecolor('white')
        
        lat_min, lat_max = region_bounds['lat']
        lon_min, lon_max = region_bounds['lon']
        
        # Find indices for this region
        lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
        lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
        
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            continue
        
        # Regional land mask and weights
        reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
        reg_lat = flux_lat[lat_idx]
        weights = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
        weights = np.broadcast_to(weights, reg_land_mask.shape)
        
        for si, (flux_scenario, prec_scenario, label, color) in enumerate(scenarios_plot):
            if flux_scenario not in flux_raw_data:
                continue
            
            data_list = flux_raw_data[flux_scenario]
            prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
            
            if not prec_scenario_files:
                continue
            
            # Compute seasonal P-ET means across all ensemble members
            all_seasonal_means = {s: [] for s in season_order}
            
            for ens_idx, et_data in enumerate(data_list):
                if ens_idx >= len(prec_scenario_files):
                    break
                
                try:
                    # Convert to ET
                    et = compute_ET_from_LHFLX(et_data)
                    et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    et_years = et.time.dt.year.values
                    et_months = et.time.dt.month.values
                    
                    # Load precipitation - use ['path'] since prec_scenario_files contains dicts
                    prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                    prec = prec_ds['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    )
                    prec_mm = prec * 86400 * 1000
                    prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                    prec_years = prec_mm.time.dt.year.values
                    prec_months = prec_mm.time.dt.month.values
                    prec_ds.close()
                    
                    # Filter to overshoot period
                    et_overshoot_mask = (et_years >= overshoot_start) & (et_years <= overshoot_end)
                    prec_overshoot_mask = (prec_years >= overshoot_start) & (prec_years <= overshoot_end)
                    
                    for season in season_order:
                        season_months = seasons[season]
                        et_season_mask = et_overshoot_mask & np.isin(et_months, season_months)
                        prec_season_mask = prec_overshoot_mask & np.isin(prec_months, season_months)
                        
                        if np.sum(et_season_mask) > 0 and np.sum(prec_season_mask) > 0:
                            et_season = np.nanmean(et_reg[et_season_mask], axis=0)
                            prec_season = np.nanmean(prec_reg[prec_season_mask], axis=0)
                            p_et = prec_season - et_season
                            
                            # Compute weighted regional mean
                            masked = np.where(reg_land_mask, p_et, np.nan)
                            w = np.where(reg_land_mask, weights, 0)
                            if np.sum(~np.isnan(masked)) > 0:
                                all_seasonal_means[season].append(
                                    np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)])
                                )
                
                except Exception as e:
                    continue
            
            # Compute ensemble mean for each season
            seasonal_means = [np.nanmean(all_seasonal_means[s]) if all_seasonal_means[s] else np.nan 
                              for s in season_order]
            
            ax.bar(x_pos + si * bar_width, seasonal_means, bar_width, 
                   label=label, color=color, alpha=0.8)
        
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
        ax.set_xlabel('Season', fontsize=16, fontweight='bold')
        ax.set_ylabel('mm/day', fontsize=16, fontweight='bold')
        ax.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos + 1.5 * bar_width)
        ax.set_xticklabels(season_order)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        if idx == 0:
            ax.legend(fontsize=8)
    
    fig.suptitle(f'Seasonal Surface Moisture Budget - African Monsoonal Regions\n(Overshoot Period {overshoot_start}-{overshoot_end})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    
    filename = 'moisture_budget_seasonal.png'
    plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Run seasonal analysis
if flux_raw_data and flux_lat is not None:
    plot_seasonal_moisture_budget(flux_raw_data, prec_files, flux_lat, flux_lon, land_mask, output_dir)
else:
    print("No data available for seasonal analysis")


In [ ]:
def plot_moisture_budget_period_comparison(moisture_budget, flux_lat, flux_lon, land_mask, output_dir):
    """Create bar plots comparing P-ET across scenarios for baseline - overshoot periods."""
    scenarios_plot = [
        ('ssp585', 'SSP5-8.5'),
        ('ssp534os', 'SSP5-3.4-OS'),
        ('ssp534os_15_feedback', 'SAI 1.5°C'),
        ('ssp534os_20_feedback', 'SAI 2.0°C'),
    ]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    fig.patch.set_facecolor('white')
    
    x_pos = np.arange(len(scenarios_plot))
    bar_width = 0.35
    
    baseline_start, baseline_end = analysis_periods['early_warming_15C']
    overshoot_start, overshoot_end = analysis_periods['overshoot_period']
    
    for idx, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
        ax = axes[idx]
        ax.set_facecolor('white')
        
        lat_min, lat_max = region_bounds['lat']
        lon_min, lon_max = region_bounds['lon']
        
        # Find indices for this region
        lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
        lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
        
        if len(lat_idx) == 0 or len(lon_idx) == 0:
            continue
        
        # Regional land mask
        reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
        reg_lat = flux_lat[lat_idx]
        weights = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
        weights = np.broadcast_to(weights, reg_land_mask.shape)
        
        baseline_values = []
        overshoot_values = []
        
        for scenario, label in scenarios_plot:
            if scenario not in moisture_budget:
                baseline_values.append(np.nan)
                overshoot_values.append(np.nan)
                continue
            
            period_data = moisture_budget[scenario]
            
            # Compute regional weighted mean for baseline
            if 'baseline' in period_data:
                baseline_full = period_data['baseline']
                baseline_reg = baseline_full[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                masked = np.where(reg_land_mask, baseline_reg, np.nan)
                w = np.where(reg_land_mask, weights, 0)
                if np.sum(~np.isnan(masked)) > 0:
                    baseline_values.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                else:
                    baseline_values.append(np.nan)
            else:
                baseline_values.append(np.nan)
            
            # Compute regional weighted mean for overshoot
            if 'overshoot' in period_data:
                overshoot_full = period_data['overshoot']
                overshoot_reg = overshoot_full[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                masked = np.where(reg_land_mask, overshoot_reg, np.nan)
                w = np.where(reg_land_mask, weights, 0)
                if np.sum(~np.isnan(masked)) > 0:
                    overshoot_values.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                else:
                    overshoot_values.append(np.nan)
            else:
                overshoot_values.append(np.nan)
        
        ax.bar(x_pos - bar_width/2, baseline_values, bar_width, 
               label=f'Baseline ({baseline_start}-{baseline_end})', color='#4292c6', alpha=0.8)
        ax.bar(x_pos + bar_width/2, overshoot_values, bar_width,
               label=f'Overshoot ({overshoot_start}-{overshoot_end})', color='#ef6548', alpha=0.8)
        
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
        ax.set_xlabel('Scenario', fontsize=16, fontweight='bold')
        ax.set_ylabel('mm/day', fontsize=16, fontweight='bold')
        ax.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([s[1] for s in scenarios_plot], rotation=15, ha='right', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        
        if idx == 0:
            ax.legend(fontsize=8)
    
    fig.suptitle('Surface Moisture Budget: Baseline - Overshoot Period\nAfrican Monsoonal Regions',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    
    filename = 'moisture_budget_period_comparison.png'
    plt.savefig(f'{output_dir}/{filename}', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Run period comparison
if moisture_budget_15C and flux_lat is not None:
    plot_moisture_budget_period_comparison(moisture_budget_15C, flux_lat, flux_lon, land_mask, output_dir)
else:
    print("No moisture budget data available for period comparison")

print("\n" + "="*60)
print("MOISTURE BUDGET ANALYSIS COMPLETE!")


In [ ]:
# MEAN ANNUAL CYCLE (MONTHLY CLIMATOLOGY)

def compute_monthly_climatology(temp_files, prec_files, scenarios_to_process, period):
    """Compute mean monthly climatology for temperature and precipitation."""
    print("\nComputing monthly climatology...")
    
    monthly_clim = {}
    months = list(range(1, 13))
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    period_start, period_end = period
    
    for scenario in scenarios_to_process:
        temp_scenario_files = [f for f in temp_files if f['scenario'] == scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == scenario]
        if not temp_scenario_files:
            continue
        print(f"  Processing {scenario}...")
        monthly_clim[scenario] = {'temp': {m: [] for m in months}, 'prec': {m: [] for m in months}}
        
        n_ens = min(len(temp_scenario_files), len(prec_scenario_files))
        for ens_idx in range(n_ens):
            try:
                temp_ds = xr.open_dataset(temp_scenario_files[ens_idx]['path'])
                if 'TREFHT' in temp_ds.data_vars:
                    tas = temp_ds['TREFHT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    ) - 273.15
                    years = tas.time.dt.year.values
                    mask = (years >= period_start) & (years <= period_end)
                    tas_period = tas.isel(time=mask)
                    for m in months:
                        month_data = tas_period.sel(time=tas_period.time.dt.month == m)
                        if len(month_data.time) > 0:
                            monthly_clim[scenario]['temp'][m].append(float(month_data.mean().values))
                temp_ds.close()
                
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                if 'PRECT' in prec_ds.data_vars:
                    pr = prec_ds['PRECT'].sel(
                        lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                        lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max'])
                    ) * 86400 * 1000
                    years = pr.time.dt.year.values
                    mask = (years >= period_start) & (years <= period_end)
                    pr_period = pr.isel(time=mask)
                    for m in months:
                        month_data = pr_period.sel(time=pr_period.time.dt.month == m)
                        if len(month_data.time) > 0:
                            monthly_clim[scenario]['prec'][m].append(float(month_data.mean().values))
                prec_ds.close()
            except Exception as e:
                print(f"    Error: {e}")
    return monthly_clim, month_names

def plot_mean_annual_cycle(monthly_clim, month_names, output_dir, period):
    """Plot mean annual cycle with 13 months (Jan repeated at end per Visioni suggestion)."""
    scenarios_plot = [
        ('ssp585', 'SSP5-8.5', '#d62728', '-'),
        ('ssp534OS', 'SSP5-3.4-OS', '#1f77b4', '-'),
        ('ssp534OS_15_feedback', 'SAI 1.5°C', '#2ca02c', '--'),
        ('ssp534OS_20_feedback', 'SAI 2.0°C', '#9467bd', '--'),
    ]
    
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.patch.set_facecolor('white')
    
    # 13 months: Jan-Dec + Jan (to complete the cycle)
    x = np.arange(1, 14)
    month_labels = month_names + ['Jan']
    
    # Temperature
    ax = axes[0]
    ax.set_facecolor('white')
    for scenario, label, color, ls in scenarios_plot:
        if scenario not in monthly_clim:
            continue
        means = [np.mean(monthly_clim[scenario]['temp'][m]) if monthly_clim[scenario]['temp'][m] else np.nan for m in range(1, 13)]
        means = means + [means[0]]  # Repeat January
        stds = [np.std(monthly_clim[scenario]['temp'][m]) if len(monthly_clim[scenario]['temp'][m]) > 1 else 0 for m in range(1, 13)]
        stds = stds + [stds[0]]
        ax.plot(x, means, color=color, linestyle=ls, linewidth=2.5, label=label, marker='o', markersize=6)
        ax.fill_between(x, np.array(means) - np.array(stds), np.array(means) + np.array(stds), color=color, alpha=0.15)
    ax.set_xlabel('Month', fontsize=14, fontweight='bold')
    ax.set_ylabel('Temperature (°C)', fontsize=14, fontweight='bold')
    ax.set_title('Mean Annual Temperature Cycle', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(fontsize=12, loc='best', framealpha=0.95, edgecolor='gray', fancybox=True)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Precipitation
    ax = axes[1]
    ax.set_facecolor('white')
    for scenario, label, color, ls in scenarios_plot:
        if scenario not in monthly_clim:
            continue
        means = [np.mean(monthly_clim[scenario]['prec'][m]) if monthly_clim[scenario]['prec'][m] else np.nan for m in range(1, 13)]
        means = means + [means[0]]  # Repeat January
        stds = [np.std(monthly_clim[scenario]['prec'][m]) if len(monthly_clim[scenario]['prec'][m]) > 1 else 0 for m in range(1, 13)]
        stds = stds + [stds[0]]
        ax.plot(x, means, color=color, linestyle=ls, linewidth=2.5, label=label, marker='o', markersize=6)
        ax.fill_between(x, np.array(means) - np.array(stds), np.array(means) + np.array(stds), color=color, alpha=0.15)
    ax.set_xlabel('Month', fontsize=14, fontweight='bold')
    ax.set_ylabel('Precipitation (mm/day)', fontsize=14, fontweight='bold')
    ax.set_title('Mean Annual Precipitation Cycle', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(fontsize=12, loc='best', framealpha=0.95, edgecolor='gray', fancybox=True)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    period_start, period_end = period
    fig.suptitle(f'Mean Annual Cycle - Africa ({period_start}-{period_end})', fontsize=18, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'{output_dir}/mean_annual_cycle.png', dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print("Saved: mean_annual_cycle.png")

# Run analysis
scenarios_for_cycle = ['ssp585', 'ssp534OS', 'ssp534OS_15_feedback', 'ssp534OS_20_feedback']
monthly_clim, month_names = compute_monthly_climatology(temp_files, prec_files, scenarios_for_cycle, analysis_periods['overshoot_period'])
if monthly_clim:
    plot_mean_annual_cycle(monthly_clim, month_names, output_dir, analysis_periods['overshoot_period'])
print("Mean annual cycle analysis complete!")


## Notes on Statistics

### Statistical Tests Used
- **Welch's t-test**: Used for pixel-wise significance testing between baseline and future periods
- Tests whether the difference in means is statistically significant (p < 0.05)
- Welch's test is used (equal_var=False) because ensemble variances may differ

### Stippling
- Black dots on difference maps indicate pixels where p < 0.05
- More stippling = more statistically significant changes

### Summary Statistics Table
- **Baseline/Future Mean**: Regional average over African land
- **Abs/Pct Change**: Difference between periods
- **Sig Area (%)**: Percentage of land area with significant change (p < 0.05)

### Optimal Growing Days (OGD) References
OGD counts days where temperature is between 10-35°C, representing suitable conditions for crop growth.

**Lower threshold (10°C):**
- FAO Crop Information (Maize): 'For germination the lowest mean daily temperature is about 10°C'
- Standard base temperature for warm-season crops (maize, sorghum, millet, rice)

**Upper threshold (35°C):**
- Lobell et al. (2011) Nature Climate Change: yield losses of 1-1.7% per degree day above 30°C in Sub-Saharan Africa
- Sánchez et al. (2014): upper optimum temperature range 30-35°C for maize
- PMC7913793: 'Maize leaf growth increases at temperatures ranging from 10 to 35°C, while it starts declining at temperatures >35°C'
- Crafts-Brandner & Law (2000): Rubisco activity highly reduced above 35°C

### Moisture Budget Analysis
- **ET (Evapotranspiration)**: Derived from latent heat flux (LHFLX)
- Formula: ET (mm/day) = LHFLX (W/m²) / 2.5×10⁶ (J/kg) × 86400 (s/day)
- Monsoonal regions analyzed: West Africa (WAF), Central Africa (CAF), East Africa (EAF)


In [ ]:
# COMBINED: Time Series (row 1) + Seasonal Bar (row 2)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

n_regions = len(MONSOONAL_REGIONS)

fig, axes = plt.subplots(2, n_regions, figsize=(7 * n_regions, 18),
                         gridspec_kw={'hspace': 0.30})
fig.patch.set_facecolor('white')

ts_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728', '-'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4', '-'),
    ('ssp585_15_feedback', 'ssp585_15_feedback', 'SSP5-8.5 1.5\u00b0C SAI', '#d62728', '--'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SSP5-3.4-OS 1.5\u00b0C SAI', '#1f77b4', '--'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SSP5-3.4-OS 2.0\u00b0C SAI', '#2ca02c', '--'),
]

bar_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SAI 1.5\u00b0C', '#2ca02c'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SAI 2.0\u00b0C', '#9467bd'),
]

seasons = {'DJF': [12, 1, 2], 'MAM': [3, 4, 5], 'JJA': [6, 7, 8], 'SON': [9, 10, 11]}
season_order = ['DJF', 'MAM', 'JJA', 'SON']
x_pos = np.arange(len(season_order))
bar_width = 0.2
overshoot_start, overshoot_end = analysis_periods['overshoot_period']

for idx, (region_name, region_bounds) in enumerate(MONSOONAL_REGIONS.items()):
    lat_min, lat_max = region_bounds['lat']
    lon_min, lon_max = region_bounds['lon']

    lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
    lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
    if len(lat_idx) == 0 or len(lon_idx) == 0:
        continue

    reg_land_mask = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
    reg_lat = flux_lat[lat_idx]
    weights = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
    weights = np.broadcast_to(weights, reg_land_mask.shape)

    # ROW 0: TIME SERIES
    ax_ts = axes[0, idx]
    ax_ts.set_facecolor('white')
    ts_all_plotted = []  # track all plotted values for ylim

    for flux_scenario, prec_scenario, label, color, ls in ts_scenarios:
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_years = None
        all_vals = []
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_ds.close()

                common_years = np.intersect1d(et_years, prec_years)
                if len(common_years) == 0:
                    continue
                p_et_ts, years_ts = [], []
                for year in common_years:
                    et_mask = et_years == year
                    prec_mask = prec_years == year
                    if np.sum(et_mask) > 0 and np.sum(prec_mask) > 0:
                        et_yr = np.nanmean(et_reg[et_mask], axis=0)
                        pr_yr = np.nanmean(prec_reg[prec_mask], axis=0)
                        p_et = pr_yr - et_yr
                        masked = np.where(reg_land_mask, p_et, np.nan)
                        w = np.where(reg_land_mask, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            p_et_ts.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                            years_ts.append(year)

                if p_et_ts:
                    years_arr = np.array(years_ts)
                    vals_arr = np.array(p_et_ts)
                    keep = years_arr <= 2099
                    if np.sum(keep) > 0:
                        if all_years is None:
                            all_years = years_arr[keep]
                        all_vals.append(vals_arr[keep])
            except:
                continue

        if all_vals:
            minlen = min(len(v) for v in all_vals)
            arr = np.array([v[:minlen] for v in all_vals])
            mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
            ax_ts.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
            ts_all_plotted.extend(mean)
            if len(arr) > 1:
                std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax_ts.fill_between(all_years[:minlen], mean - std, mean + std, color=color, alpha=0.15)
                ts_all_plotted.extend(mean - std)
                ts_all_plotted.extend(mean + std)

    # Auto-scale y-axis close to data range
    if ts_all_plotted:
        data_min = np.nanmin(ts_all_plotted)
        data_max = np.nanmax(ts_all_plotted)
        margin = (data_max - data_min) * 0.08
        ax_ts.set_ylim(data_min - margin, data_max + margin)

    ax_ts.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax_ts.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=16, fontweight='bold')
    ax_ts.set_xlim(2015, 2100)
    ax_ts.tick_params(labelsize=12)
    ax_ts.grid(True, alpha=0.3, linestyle='--')
    if idx == 0:
        ax_ts.set_ylabel('(mm/day)', fontsize=18, fontweight='bold')
        ax_ts.legend(fontsize=7, loc='best', framealpha=0.3)

    # ROW 1: SEASONAL BAR
    ax_bar = axes[1, idx]
    ax_bar.set_facecolor('white')

    for si, (flux_scenario, prec_scenario, label, color) in enumerate(bar_scenarios):
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_seasonal_means = {s: [] for s in season_order}
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                et_months = et.time.dt.month.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_months = prec_mm.time.dt.month.values
                prec_ds.close()

                et_ov = (et_years >= overshoot_start) & (et_years <= overshoot_end)
                prec_ov = (prec_years >= overshoot_start) & (prec_years <= overshoot_end)

                for season in season_order:
                    sm = seasons[season]
                    et_sm = et_ov & np.isin(et_months, sm)
                    prec_sm = prec_ov & np.isin(prec_months, sm)
                    if np.sum(et_sm) > 0 and np.sum(prec_sm) > 0:
                        et_s = np.nanmean(et_reg[et_sm], axis=0)
                        pr_s = np.nanmean(prec_reg[prec_sm], axis=0)
                        p_et = pr_s - et_s
                        masked = np.where(reg_land_mask, p_et, np.nan)
                        w = np.where(reg_land_mask, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            all_seasonal_means[season].append(
                                np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
            except:
                continue

        seasonal_vals = [np.nanmean(all_seasonal_means[s]) if all_seasonal_means[s] else np.nan
                         for s in season_order]
        ax_bar.bar(x_pos + si * bar_width, seasonal_vals, bar_width,
                   label=label, color=color, alpha=0.8)

    ax_bar.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax_bar.set_xlabel('Season', fontsize=16, fontweight='bold')
    ax_bar.set_xticks(x_pos + 1.5 * bar_width)
    ax_bar.set_xticklabels(season_order, fontsize=12)
    ax_bar.tick_params(labelsize=12)
    ax_bar.grid(True, alpha=0.3, axis='y', linestyle='--')
    if idx == 0:
        ax_bar.set_ylabel('(mm/day)', fontsize=18, fontweight='bold')
        ax_bar.legend(fontsize=8, framealpha=0.3)

fig.suptitle('Surface Moisture Budget: African Monsoonal Regions',
             fontsize=22, fontweight='bold')
plt.tight_layout(rect=[0.02, 0, 1, 0.95])

fname = f'{output_dir}/moisture_budget_combined_ts_seasonal.png'
plt.savefig(fname, dpi=700, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname}')


In [ ]:
# Two figures: 2x2 Time Series + 2x2 Seasonal Bar
# Run after the spatial moisture budget cell

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

ts_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728', '-'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4', '-'),
    ('ssp585_15_feedback', 'ssp585_15_feedback', 'SSP5-8.5 1.5°C SAI', '#d62728', '--'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SSP5-3.4-OS 1.5°C SAI', '#1f77b4', '--'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SSP5-3.4-OS 2.0°C SAI', '#2ca02c', '--'),
]

bar_scenarios = [
    ('ssp585', 'ssp585', 'SSP5-8.5', '#d62728'),
    ('ssp534os', 'ssp534OS', 'SSP5-3.4-OS', '#1f77b4'),
    ('ssp534os_15_feedback', 'ssp534OS_15_feedback', 'SAI 1.5°C', '#2ca02c'),
    ('ssp534os_20_feedback', 'ssp534OS_20_feedback', 'SAI 2.0°C', '#9467bd'),
]

seasons = {'DJF': [12, 1, 2], 'MAM': [3, 4, 5], 'JJA': [6, 7, 8], 'SON': [9, 10, 11]}
season_order = ['DJF', 'MAM', 'JJA', 'SON']
x_pos = np.arange(len(season_order))
bar_width = 0.2
overshoot_start, overshoot_end = analysis_periods['overshoot_period']

regions = list(MONSOONAL_REGIONS.items())

# Helper to get regional indices and masks
def get_region_data(region_bounds):
    lat_min, lat_max = region_bounds['lat']
    lon_min, lon_max = region_bounds['lon']
    lat_idx = np.where((flux_lat >= lat_min) & (flux_lat <= lat_max))[0]
    lon_idx = np.where((flux_lon >= lon_min) & (flux_lon <= lon_max))[0]
    if len(lat_idx) == 0 or len(lon_idx) == 0:
        return None
    reg_land = land_mask[lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
    reg_lat = flux_lat[lat_idx]
    w = np.cos(np.deg2rad(reg_lat))[:, np.newaxis]
    w = np.broadcast_to(w, reg_land.shape)
    return lat_idx, lon_idx, reg_land, w

# FIGURE 1: 2x2 TIME SERIES
fig1, axes1 = plt.subplots(2, 2, figsize=(16, 12))
fig1.patch.set_facecolor('white')
axes1_flat = axes1.flatten()

for idx, (region_name, region_bounds) in enumerate(regions):
    rd = get_region_data(region_bounds)
    if rd is None:
        continue
    lat_idx, lon_idx, reg_land, weights = rd
    ax = axes1_flat[idx]
    ax.set_facecolor('white')
    ts_all = []

    for flux_scenario, prec_scenario, label, color, ls in ts_scenarios:
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_years = None
        all_vals = []
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_ds.close()

                common_years = np.intersect1d(et_years, prec_years)
                if len(common_years) == 0:
                    continue
                p_et_ts, years_ts = [], []
                for year in common_years:
                    em = et_years == year
                    pm = prec_years == year
                    if np.sum(em) > 0 and np.sum(pm) > 0:
                        et_yr = np.nanmean(et_reg[em], axis=0)
                        pr_yr = np.nanmean(prec_reg[pm], axis=0)
                        pet = pr_yr - et_yr
                        masked = np.where(reg_land, pet, np.nan)
                        w = np.where(reg_land, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            p_et_ts.append(np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
                            years_ts.append(year)
                if p_et_ts:
                    ya = np.array(years_ts)
                    va = np.array(p_et_ts)
                    keep = ya <= 2099
                    if np.sum(keep) > 0:
                        if all_years is None:
                            all_years = ya[keep]
                        all_vals.append(va[keep])
            except:
                continue

        if all_vals:
            minlen = min(len(v) for v in all_vals)
            arr = np.array([v[:minlen] for v in all_vals])
            mean = pd.Series(np.nanmean(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
            ax.plot(all_years[:minlen], mean, color=color, linewidth=2, label=label, linestyle=ls)
            ts_all.extend(mean)
            if len(arr) > 1:
                std = pd.Series(np.nanstd(arr, axis=0)).rolling(5, center=True, min_periods=1).mean().values
                ax.fill_between(all_years[:minlen], mean - std, mean + std, color=color, alpha=0.15)
                ts_all.extend(mean - std)
                ts_all.extend(mean + std)

    if ts_all:
        dmin, dmax = np.nanmin(ts_all), np.nanmax(ts_all)
        margin = (dmax - dmin) * 0.08
        ax.set_ylim(dmin - margin, dmax + margin)

    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=15, fontweight='bold')
    ax.set_xlim(2015, 2100)
    ax.set_ylabel('(mm/day)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year', fontsize=14, fontweight='bold')
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')
    if idx == 0:
        ax.legend(fontsize=8, loc='best', framealpha=0.3)

fig1.suptitle('Surface Moisture Budget: Time Series\n(5-year running mean, shading: \u00b11 std across ensembles)',
              fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
fname1 = f'{output_dir}/moisture_budget_timeseries_2x2.png'
plt.savefig(fname1, dpi=700, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname1}')

# FIGURE 2: 2x2 SEASONAL BAR
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 12))
fig2.patch.set_facecolor('white')
axes2_flat = axes2.flatten()

for idx, (region_name, region_bounds) in enumerate(regions):
    rd = get_region_data(region_bounds)
    if rd is None:
        continue
    lat_idx, lon_idx, reg_land, weights = rd
    ax = axes2_flat[idx]
    ax.set_facecolor('white')

    for si, (flux_scenario, prec_scenario, label, color) in enumerate(bar_scenarios):
        if flux_scenario not in flux_raw_data:
            continue
        data_list = flux_raw_data[flux_scenario]
        prec_scenario_files = [f for f in prec_files if f['scenario'] == prec_scenario]
        if not prec_scenario_files:
            continue

        all_seasonal_means = {s: [] for s in season_order}
        for ens_idx, et_data in enumerate(data_list):
            if ens_idx >= len(prec_scenario_files):
                break
            try:
                et = compute_ET_from_LHFLX(et_data)
                et_reg = et.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                et_years = et.time.dt.year.values
                et_months = et.time.dt.month.values
                prec_ds = xr.open_dataset(prec_scenario_files[ens_idx]['path'])
                prec = prec_ds['PRECT'].sel(
                    lat=slice(africa_bounds['lat_min'], africa_bounds['lat_max']),
                    lon=slice(africa_bounds['lon_min'], africa_bounds['lon_max']))
                prec_mm = prec * 86400 * 1000
                prec_reg = prec_mm.values[:, lat_idx[0]:lat_idx[-1]+1, lon_idx[0]:lon_idx[-1]+1]
                prec_years = prec_mm.time.dt.year.values
                prec_months = prec_mm.time.dt.month.values
                prec_ds.close()

                et_ov = (et_years >= overshoot_start) & (et_years <= overshoot_end)
                prec_ov = (prec_years >= overshoot_start) & (prec_years <= overshoot_end)

                for season in season_order:
                    sm = seasons[season]
                    et_sm = et_ov & np.isin(et_months, sm)
                    prec_sm = prec_ov & np.isin(prec_months, sm)
                    if np.sum(et_sm) > 0 and np.sum(prec_sm) > 0:
                        et_s = np.nanmean(et_reg[et_sm], axis=0)
                        pr_s = np.nanmean(prec_reg[prec_sm], axis=0)
                        pet = pr_s - et_s
                        masked = np.where(reg_land, pet, np.nan)
                        w = np.where(reg_land, weights, 0)
                        if np.sum(~np.isnan(masked)) > 0:
                            all_seasonal_means[season].append(
                                np.nansum(masked * w) / np.nansum(w[~np.isnan(masked)]))
            except:
                continue

        seasonal_vals = [np.nanmean(all_seasonal_means[s]) if all_seasonal_means[s] else np.nan
                         for s in season_order]
        ax.bar(x_pos + si * bar_width, seasonal_vals, bar_width,
               label=label, color=color, alpha=0.8)

    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.set_title(f'{region_name} ({region_bounds["abbrev"]})', fontsize=15, fontweight='bold')
    ax.set_xlabel('Season', fontsize=14, fontweight='bold')
    ax.set_ylabel('(mm/day)', fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos + 1.5 * bar_width)
    ax.set_xticklabels(season_order, fontsize=12)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    if idx == 0:
        ax.legend(fontsize=9, framealpha=0.3)

fig2.suptitle(f'Seasonal Surface Moisture Budget \nOvershoot Period ({overshoot_start}\u2013{overshoot_end})',
              fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
fname2 = f'{output_dir}/moisture_budget_seasonal_2x2.png'
plt.savefig(fname2, dpi=700, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {fname2}')
